# AeroSync-Mamba v32 — Expert training (Section 0), three-way complementarity (Part 0) and a learnable decision router over Audio / IMU / Fusion (Part 1)

**This notebook trains its own experts.** Version 29's saved per-window probabilities no longer exist. **Section 0
therefore trains A, I and F1 itself**: 5 day-out folds × seeds 42 and 123 = 30 runs on a GPU with real `mamba_ssm`.
It writes their per-window test probabilities to `/kaggle/working/v32_npz/{model}_{fold}_{seed}.npz`.

**Section 0 is version 29's code, not a reconstruction.** Its data, model and training cells are v29's cells,
**byte-for-byte**. Before any of them runs, the notebook reads `version 29/minor-project-v29.ipynb` from the project
repo and checks that each ported cell is identical to v29's source. If the file is missing, unreadable or different,
the notebook stops before any architecture code runs.

The only additions are small override cells. They redirect output paths to `v32_*` and restrict the model list to
A, I and F1, adding D, F0 and F2 only if `TRAIN_EXTRA_MODELS = True`.

**Parts 0 and 1 are unchanged in logic.** They read the `.npz` files Section 0 has just written, looking in
`/kaggle/working/v32_npz/` first and `version 29/Npz/` second. After Section 0 the experts are fixed, and the only
model trained in Parts 0–1 is the small router MLP, which works on those saved probabilities.

## Section 0 — Train the experts A, I and F1 with version 29's code
- **0.1** Locate the project repo, read `version 29/minor-project-v29.ipynb`, and verify that the ported cells are
  byte-identical. **The notebook stops here if the file cannot be read.**
- **0.2** v29's environment and dataset cell, then v30-style path overrides to `v32_*`.
- **0.3** v29's `mamba_ssm` install and import: real `mamba_ssm` only, no fallback.
- **0.4** v29's constants, then the model-list override (`TRAIN_EXTRA_MODELS`), and v29's seeding.
- **0.5** v29's parse, metadata, paired-window and fold cells, with v29's integrity hooks, plus assertions against
  v29's recorded window hash and per-fold z-score hashes.
- **0.6** v29's model classes (D verbatim from v24; A, I, F0, F1, F2), the parameter table and the Mamba-config hook.
- **0.7** v29's training function and resumable sweep: fold-major, then A, I, F1, then seed. Each run is logged to
  `v32_expert_training_results.csv`.
- **0.8** Confirm that all expert `.npz` files exist, then a plausibility check against the prior v29 numbers.

In [1]:
# ==============================================================================
# 0.1  READ version 29/minor-project-v29.ipynb AND VERIFY THE PORTED CELLS ARE BYTE-IDENTICAL (stops if missing/different)
# ==============================================================================
import hashlib, json, subprocess, time
from pathlib import Path

T0_SECTION0 = time.time()
PORTED_V29_CELLS = {"3": "e95708d73037092879bb76eee853c560cb84c85d", "4": "44af7fce9f4a0d69d3b0dd86d65187afd8c6638d", "5": "a7a2e4e3291c57c8fd1420646af576ab47a4366f", "6": "5558969b2a20b641f33e943c555a14374d92d095", "8": "7de45a38532aac8ca558b6ddb3e3b85c9580c3d8", "9": "5acf7149244078d98b41ca51faab20175d6a3f8c", "10": "441980d6287757d1337ff61331e8ad781dc9dcc7", "12": "83b40f16d31d1d88f3b2f9eb61e73bac7d859a0c", "14": "242155131606406255335a5afdbefb927b073f12", "15": "ad3cd90ad102c6da280e9e581ea126ae685f78e1", "16": "1734825f421f9ff1b02d481322dc3570d8042192", "18": "729d9885249eff8ad8e85f2c4d39ba3b9e96555c", "22": "974e6537ff94a6b7e86c8fc4f0fcbfc4636b9fad"}          # {v29 cell index: sha1 of its source}, recorded when this notebook was written


def _has_v29(root):
    return (root / "version 29" / "minor-project-v29.ipynb").is_file()


_work = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "v32_outputs"
_work.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = next((r for r in (Path.cwd().parent, Path.cwd()) if _has_v29(r)), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = _work / "AeroSync-Mamba"
    if not (PROJECT_ROOT / ".git").exists():
        print("Cloning the public project repo (--depth 1) to read version 29's notebook ...")
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Sainava/AeroSync-Mamba.git", str(PROJECT_ROOT)],
                       check=False)
V29_NOTEBOOK = PROJECT_ROOT / "version 29" / "minor-project-v29.ipynb"
try:
    _v29 = json.loads(V29_NOTEBOOK.read_text(encoding="utf-8"))
except Exception as exc:
    print("🚨" * 50)
    print(f"🚨  version 29's notebook could not be read.\n🚨  Expected: {V29_NOTEBOOK}\n🚨  Error: {exc!r}\n"
          f"🚨  STOPPING before any architecture code: the expert models are ported from this file and are never rebuilt "
          f"from a description.")
    print("🚨" * 50)
    raise RuntimeError(f"version 29 notebook not found/readable at {V29_NOTEBOOK} — stopped before any architecture reconstruction")
_mismatch = {}
for idx, expected in PORTED_V29_CELLS.items():
    src = _v29["cells"][int(idx)]["source"]
    got = hashlib.sha1(("".join(src) if isinstance(src, list) else src).encode()).hexdigest()
    if got != expected:
        _mismatch[idx] = (expected[:12], got[:12])
if _mismatch:
    raise RuntimeError(f"{V29_NOTEBOOK} differs from the cells ported into this notebook: {_mismatch} — stopping "
                       "(the ported code must equal version 29's source).")
print(f"[CHECK] version 29 notebook read: {V29_NOTEBOOK}")
print(f"[CHECK] {len(PORTED_V29_CELLS)} ported cells are byte-identical to v29's cells {sorted(int(i) for i in PORTED_V29_CELLS)}")

Cloning the public project repo (--depth 1) to read version 29's notebook ...


Cloning into '/kaggle/working/AeroSync-Mamba'...


[CHECK] version 29 notebook read: /kaggle/working/AeroSync-Mamba/version 29/minor-project-v29.ipynb
[CHECK] 13 ported cells are byte-identical to v29's cells [3, 4, 5, 6, 8, 9, 10, 12, 14, 15, 16, 18, 22]


In [2]:
# ==============================================================================
# 2a. ENVIRONMENT, PATHS, DATASET
# ==============================================================================
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"    # cuBLAS reproducibility; only effective before CUDA initializes

import ast, copy, csv, gc, glob, hashlib, json, math, random, re, subprocess, sys, time, warnings
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or Path("/content").exists())
WORK = Path("/kaggle/working") if IS_KAGGLE else (Path("/content") if IS_COLAB else Path.cwd() / "v29_outputs")
BASE = WORK / "tii"
WORK.mkdir(parents=True, exist_ok=True)
BASE.mkdir(parents=True, exist_ok=True)
REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"


def _is_dataset_root(p):
    return p.is_dir() and all((p / str(c)).is_dir() for c in range(5))


_att = []
for _pat in ("/kaggle/input/*/Dataset", "/kaggle/input/*/*/Dataset", "/kaggle/input/*/*/*/Dataset"):
    _att += [Path(p) for p in sorted(glob.glob(_pat)) if _is_dataset_root(Path(p))]
if _att:
    DATA_ROOT = _att[0]
else:
    _repo = BASE / "UAV-Realistic-Fault-Dataset"
    if not _is_dataset_root(_repo / "Dataset"):
        print(f"Cloning {REPO_URL} (--depth 1) ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True)
    DATA_ROOT = _repo / "Dataset"
assert _is_dataset_root(DATA_ROOT)

CACHE_DIR = WORK / "v29_cache"
PROBS_DIR = WORK / "v29_probs"
CKPT_DIR = WORK / "v29_checkpoints"
RESULTS_CSV = WORK / "gated_fusion_results.csv"
TABLE_CSV = WORK / "v29_results_table.csv"
for _d in (CACHE_DIR, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print(f"DATA_ROOT {DATA_ROOT} | results {RESULTS_CSV} | probs {PROBS_DIR} | checkpoints {CKPT_DIR}")

Cloning https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git (--depth 1) ...


Cloning into '/kaggle/working/tii/UAV-Realistic-Fault-Dataset'...


DATA_ROOT /kaggle/working/tii/UAV-Realistic-Fault-Dataset/Dataset | results /kaggle/working/gated_fusion_results.csv | probs /kaggle/working/v29_probs | checkpoints /kaggle/working/v29_checkpoints


Updating files: 100% (217/217), done.


In [3]:
# ==============================================================================
# 0.2b  v32 OVERRIDES of v29's output paths (v29's own folders are left unused)
# ==============================================================================
PROBS_DIR = WORK / "v32_npz"                               # the .npz files Parts 0 and 1 read
CKPT_DIR = WORK / "v32_checkpoints"
RESULTS_CSV = WORK / "v32_expert_training_results.csv"
TABLE_CSV = WORK / "v32_expert_results_table.csv"
for _d in (PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print(f"v32 outputs: probs {PROBS_DIR} | checkpoints {CKPT_DIR} | run log {RESULTS_CSV}")

v32 outputs: probs /kaggle/working/v32_npz | checkpoints /kaggle/working/v32_checkpoints | run log /kaggle/working/v32_expert_training_results.csv


In [4]:
# ==============================================================================
# 2b. REAL mamba-ssm — install if missing, import, raise on failure (no fallback)
# ==============================================================================
import importlib, importlib.util
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU: real mamba-ssm needs a GPU. Enable a GPU accelerator and re-run.")
if importlib.util.find_spec("mamba_ssm") is None:
    _ok = (_pip(["packaging", "ninja"]) and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output). Check internet + GPU, restart, re-run.")
try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"`from mamba_ssm import Mamba` failed: {exc!r}. No fallback is used.") from exc
device = torch.device("cuda")
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} ({MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__}) | torch "
      f"{torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

  pip install packaging ninja
  pip install causal-conv1d>=1.2.0 --no-build-isolation
  pip install mamba-ssm --no-build-isolation
mamba_ssm 2.3.2.post1 (mamba_ssm.modules.mamba_simple.Mamba) | torch 2.10.0+cu128 | CUDA 12.8 | GPU Tesla T4


In [6]:
# ==============================================================================
# 2c. CONSTANTS — every fixed value lives here
# ==============================================================================
# --- run matrix ---
SEEDS_REQUIRED = [42, 123]
EXTRA_SEED_2024 = False          # OPTIONAL extension: set True and re-run; logged runs are skipped (resumable)
SEEDS = SEEDS_REQUIRED + ([2024] if EXTRA_SEED_2024 else [])
RUN_F2 = True                    # SECONDARY ablation — skippable; runs after F1 within each fold
MODELS = ["A", "I", "D", "F0", "F1"] + (["F2"] if RUN_F2 else [])
FUSED_MODELS = {"D", "F0", "F1", "F2"}
NEW_MODELS = ["F0", "F1"] + (["F2"] if RUN_F2 else [])

# --- missions / days (v23/v24/v27; asserted in section 1) ---
N_CLASSES = 5
EXCLUDE_RATIO = 0.5
EXPECTED_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                     "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = 95
EXPECTED_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"
STEADY = (0.10, 0.90)

# --- inner validation ---
INNER_VAL_RULE = "fallback_2026"   # "fallback_2026" (standalone audio/IMU runs; spec) or "v27" (v27/v28's rule, seed 2027)
INNER_VAL_FRACTION = 0.15
INNER_VAL_FALLBACK_SEED = 2026
INNER_VAL_SEED_V27 = 2027
assert INNER_VAL_RULE in ("fallback_2026", "v27")

# --- audio features (audio notebook == v28 A1) ---
N_SLOTS, DROP_SLOT, FRAME = 6, 5, 256
KEEP_SLOTS = [s for s in range(N_SLOTS) if s != DROP_SLOT]
BUFFER_LEN = N_SLOTS * FRAME                     # 1536
KEEP_BINS = list(range(1, 17))
N_AUDIO_FEAT = len(KEEP_SLOTS) * len(KEEP_BINS)  # 80
AUDIO_STD_EPS = 1e-6

# --- paired windows (v24 CORRECTED_ALIGNMENT) ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
TA_AUDIO = 64
TS_SENSOR = 128
MIN_SENSOR_SAMPLES_PER_WINDOW = 10
N_SENSOR_CHANNELS = 6
SENSOR_STD_EPS = 1e-6

# --- Condition D (v22/v24/v27, unchanged) ---
N_AUDIO_BINS = N_AUDIO_FEAT      # D's audio input width: v24 used 64 mono bins; the shared protocol feeds the 80 A1 features
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                  # used only if the lag bias were ON; Condition D -> beta = 0.0
LAG_BIAS_HIDDEN = 32
LAG_BIAS_PARAMS = 97
LAG_BIAS_LN_EPS = 1e-5
HTT_TIME_EMBED_ENABLED = False
LACA_LAG_BIAS_ENABLED = False
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496
DROPOUT = 0.15
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- new encoders / gates ---
IMU_PRECONV_KERNEL = 7           # one Conv1d, 'same' padding; see the design notes for the choice
IMU_PRECONV_CHANNELS = 6         # keeps the existing projection's input width (6) -> existing projection reused unchanged
GATE_HIDDEN = 64                 # F1/F2 gate MLP: Linear(512 -> 64) -> GELU -> Linear(64 -> 2 or 1)

# --- optimization (v24 == v28) ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- analysis (fixed before any result) ---
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
NOISE_FLOOR_F1 = 0.016           # project's seed-to-seed SD of macro-F1 (v15/v16)
GATE_STD_THRESHOLD = 0.05        # a gate whose across-window SD on the test set is below this is "near-constant"
N_HASH_SAMPLE = 25

# --- determinism investigation ---
DET_FOLD, DET_SEED, DET_EPOCHS = 1, 42, 2

print(f"Models {MODELS} | seeds {SEEDS} (EXTRA_SEED_2024={EXTRA_SEED_2024}) | RUN_F2={RUN_F2} | "
      f"{len(MODELS) * len(DAYS) * len(SEEDS)} runs | inner validation: {INNER_VAL_RULE}")

Models ['A', 'I', 'D', 'F0', 'F1', 'F2'] | seeds [42, 123] (EXTRA_SEED_2024=False) | RUN_F2=True | 60 runs | inner validation: fallback_2026


In [7]:
# ==============================================================================
# 0.4b  v32 OVERRIDE of v29's model list: A, I, F1 (required by Parts 0/1); D, F0, F2 only if TRAIN_EXTRA_MODELS
# ==============================================================================
TRAIN_EXTRA_MODELS = False
MODELS = ["A", "I", "F1"] + (["D", "F0", "F2"] if TRAIN_EXTRA_MODELS else [])
SEEDS = [42, 123]
print("=" * 100)
print(f"TRAIN_EXTRA_MODELS = {TRAIN_EXTRA_MODELS} -> training {MODELS} x {len(DAYS)} folds x seeds {SEEDS} = "
      f"{len(MODELS) * len(DAYS) * len(SEEDS)} runs")
if not TRAIN_EXTRA_MODELS:
    print("D, F0, F2 are NOT trained in this session. Any value shown for them is a HARDCODED REFERENCE VALUE, NOT REPRODUCED "
          "IN THIS SESSION, from a prior run.")
print("=" * 100)
# Reference values from version 29's run (pooled day-out macro-F1; window, mission) — plausibility check / context ONLY,
# never used in any computed statistic, CI or paired test.
PRIOR_V29 = {"A": (0.795, 0.864), "I": (0.893, 0.968), "F1": (0.869, 0.894),
             "D": (0.871, 0.893), "F0": (0.838, 0.873), "F2": (0.853, 0.883)}
PLAUSIBILITY_FACTOR = 3                                     # flag |delta| > 3 x NOISE_FLOOR_F1 (= 0.048)

TRAIN_EXTRA_MODELS = False -> training ['A', 'I', 'F1'] x 5 folds x seeds [42, 123] = 30 runs
D, F0, F2 are NOT trained in this session. Any value shown for them is a HARDCODED REFERENCE VALUE, NOT REPRODUCED IN THIS SESSION, from a prior run.


In [8]:
# ==============================================================================
# 2d. SEEDING, DETERMINISM FLAGS, HOOK HELPER
# ==============================================================================
def seed_all(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)   # section 5 switches it off and on to test its effect
seed_all(42)

HOOKS = defaultdict(list)


def hook(name, ok, msg, stop=True):
    ok = bool(ok)
    HOOKS[name].append((ok, msg))
    print(f"[HOOK {name}] {'PASS' if ok else 'FAIL'} — {msg}")
    if not ok and stop:
        raise RuntimeError(f"HOOK {name} FAILED: {msg}")
    return ok


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


def sha16(*arrays):
    h = hashlib.sha1()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]

### 0.5 Data — v29's cells, verbatim
v29's own hooks run inside these cells: the exclusion set, 95 missions, class counts [18, 19, 18, 20, 20], day counts
[10, 36, 17, 15, 17], slot 6 zero, the pairing span and hash, and the fold partition. After them, two assertions
compare the rebuilt windows and folds with the values v29 recorded:
- the paired-window content sha1 is `44c04ca18ec7a131` (5238 windows);
- every fold's z-score hash and window counts match v29's.

In [9]:
# ==============================================================================
# 3a. PARSE (all 99 folders: the 4 exclusions are computed from metadata, not assumed) -> v29_cache/<mission>.npz
# ==============================================================================
def unwrap_message(obj):                                   # IMU notebook, unchanged
    if isinstance(obj, dict):
        for key in ("msg", "message", "data"):
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj


def get_vec(d, candidates):                                # IMU notebook, unchanged
    for key in candidates:
        if key in d:
            v = d[key]
            if isinstance(v, (list, tuple)) and len(v) >= 3:
                return np.asarray(v[:3], dtype=np.float64)
        vals, ok = [], True
        for i in range(3):
            k = key.format(i=i)
            if k in d:
                vals.append(d[k])
            else:
                ok = False
                break
        if ok:
            return np.asarray(vals, dtype=np.float64)
    return None


def parse_imu_file(path):
    """IMU notebook's parser: rows [timestamp_us, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z] in FILE ORDER."""
    rows, bad = [], 0
    with open(path, "r") as f:
        for line in f:
            try:
                obj = unwrap_message(json.loads(line))
                ts = obj.get("timestamp", obj.get("time_stamp", None))
                acc = get_vec(obj, ["accelerometer_m_s2", "accelerometer_m_s2[{i}]", "accelerometer", "acceleration"])
                gyro = get_vec(obj, ["gyro_rad", "gyro_rad[{i}]", "gyro", "angular_velocity"])
                if ts is None or acc is None or gyro is None:
                    bad += 1
                    continue
                rows.append([float(ts), *acc.tolist(), *gyro.tolist()])
            except Exception:
                bad += 1
    return np.asarray(rows, dtype=np.float64).reshape(-1, 7), bad


def parse_mission(mdir):
    cache = CACHE_DIR / f"{mdir.name}.npz"
    if cache.exists():
        z = np.load(cache)
        return {k: z[k] for k in z.files}
    s_path = sorted(mdir.glob("*SensorCombined*.jsonl"))[0]
    a_path = sorted(mdir.glob("*AudioBuffer*.jsonl"))[0]
    imu, bad = parse_imu_file(s_path)
    chunks, lens, rst, rsts = [], Counter(), None, set()
    with open(a_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            x = rec.get("data")
            if x is None or len(x) == 0:
                continue
            if "recording_started_time" in rec:
                rsts.add(float(rec["recording_started_time"]))
                if rst is None:
                    rst = float(rec["recording_started_time"])          # v24: the FIRST record's value
            lens[len(x)] += 1
            chunks.append(x)
    modal = lens.most_common(1)[0][0]
    assert modal == BUFFER_LEN, f"{mdir.name}: modal buffer length {modal}"
    assert rst is not None, f"{mdir.name}: no recording_started_time — the audio cannot be anchored"
    keep_idx = np.array([i for i, x in enumerate(chunks) if len(x) == modal], dtype=np.int64)
    raw = np.asarray([chunks[i] for i in keep_idx], dtype=np.int64)
    n_all = len(chunks)
    del chunks
    frames = raw.reshape(len(raw), FRAME, N_SLOTS)                # sample i -> frame i // 6, slot i % 6
    slot6_zero = bool(np.all(frames[:, :, DROP_SLOT] == 0))
    ch = frames[:, :, KEEP_SLOTS].astype(np.float64)
    ch = ch - ch.mean(axis=1, keepdims=True)                      # per-chunk mean removed, per channel
    pw = (np.abs(np.fft.rfft(ch, axis=1)) ** 2).transpose(0, 2, 1).astype(np.float32)    # (n, 5, 129)
    feats = (pw[:, :, KEEP_BINS] ** 0.25).reshape(len(pw), N_AUDIO_FEAT).astype(np.float32)   # sqrt|rFFT|, channel-major
    out = {"imu_t": imu[:, 0], "imu_x": imu[:, 1:].astype(np.float32), "imu_bad": bad, "feats": feats,
           "keep_idx": keep_idx, "n_all_chunks": n_all, "rst": rst, "n_rst_values": len(rsts), "slot6_zero": slot6_zero}
    tmp = CACHE_DIR / f"{mdir.name}.tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, cache)
    return out


def mission_day(name):
    d = name[8:18].replace("_", "-")
    return DAY_MERGE.get(d, d)


ALL_DIRS, LABEL_OF = [], {}
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not (class_dir.is_dir() and class_dir.name.isdigit()):
        continue
    for mdir in sorted(class_dir.iterdir()):
        if mdir.is_dir() and not mdir.name.startswith("."):
            ALL_DIRS.append(mdir)
            LABEL_OF[mdir.name] = int(class_dir.name)
_t0 = time.time()
PARSED = {}
for i, mdir in enumerate(ALL_DIRS, 1):
    PARSED[mdir.name] = parse_mission(mdir)
    if i % 10 == 0 or i == len(ALL_DIRS):
        print(f"  parsed {i}/{len(ALL_DIRS)} missions | {time.time() - _t0:.0f} s")

  parsed 10/99 missions | 17 s
  parsed 20/99 missions | 35 s
  parsed 30/99 missions | 55 s
  parsed 40/99 missions | 72 s
  parsed 50/99 missions | 90 s
  parsed 60/99 missions | 109 s
  parsed 70/99 missions | 127 s
  parsed 80/99 missions | 146 s
  parsed 90/99 missions | 165 s
  parsed 99/99 missions | 180 s


In [10]:
# ==============================================================================
# 3b. METADATA — v24's corrected audio anchor, exclusion rule, mission / class / day counts (HOOK data)
# ==============================================================================
def timebase(P):
    """v24 CORRECTED_ALIGNMENT time axes. IMU: FILE ORDER, seconds from the first sample. Audio: anchored at
    recording_started_time; chunk duration = (mission duration - offset) / number of chunks."""
    t_us = P["imu_t"]
    s_times = (t_us - t_us[0]) / 1e6
    duration = float(s_times[-1] - s_times[0])
    offset = (float(P["rst"]) - float(t_us[0])) / 1e6
    chunk = (duration - offset) / int(P["n_all_chunks"])
    a_times = (offset + np.arange(int(P["n_all_chunks"]), dtype=np.float64) * chunk)[P["keep_idx"]]
    return s_times, a_times, duration, offset, chunk


META = []
for name, P in PARSED.items():
    s_times, a_times, dur, off, ch = timebase(P)
    META.append({"mission": name, "class": LABEL_OF[name], "day": mission_day(name), "duration_s": dur, "offset_s": off,
                 "ratio": off / dur, "chunk_ms": 1e3 * ch, "n_imu": len(s_times), "imu_inversions": int((np.diff(s_times) < 0).sum()),
                 "n_chunks": int(P["n_all_chunks"]), "chunks_dropped": int(P["n_all_chunks"]) - len(P["keep_idx"]),
                 "rst_values": int(P["n_rst_values"]), "slot6_zero": bool(P["slot6_zero"])})
META = pd.DataFrame(META).set_index("mission")
EXCLUDED = set(META.index[META.ratio >= EXCLUDE_RATIO])
KEPT = sorted(set(META.index) - EXCLUDED)
M = META.loc[KEPT]
LABEL = {n: int(M.loc[n, "class"]) for n in KEPT}
DAY = {n: M.loc[n, "day"] for n in KEPT}
_cls = np.bincount(M["class"], minlength=N_CLASSES).tolist()
_days = [int((M.day == d).sum()) for d in DAYS]
print(META.loc[sorted(EXCLUDED), ["class", "day", "offset_s", "duration_s", "ratio"]].round(3).to_string())
print(f"\nKept {len(KEPT)} missions | max kept ratio {M.ratio.max():.3f} | median offset {M.offset_s.median():.2f} s | "
      f"chunk duration {M.chunk_ms.median():.2f} ms (median; v24's corrected-anchor formula) | "
      f"non-modal chunks dropped {int(M.chunks_dropped.sum())} | missions with IMU timestamp inversions "
      f"{int((M.imu_inversions > 0).sum())} (total {int(M.imu_inversions.sum())})")
print(pd.crosstab(M.day, M["class"]).reindex(DAYS).to_string())
hook("data", EXCLUDED == EXPECTED_EXCLUDED, f"exclusion set == the 4 known names: {sorted(EXCLUDED)}")
hook("data", len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and _days == EXPECTED_DAY_COUNTS,
     f"{len(KEPT)} missions, classes {_cls}, days {_days} (expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS}, {EXPECTED_DAY_COUNTS})")
hook("data", bool(M.slot6_zero.all()) and bool((M.rst_values == 1).all()),
     f"audio slot 6 identically zero in {int(M.slot6_zero.sum())}/{len(M)}; recording_started_time constant in "
     f"{int((M.rst_values == 1).sum())}/{len(M)}")

                             class         day  offset_s  duration_s  ratio
mission                                                                    
rosbag2_2021_12_29-09_10_31      1  2021-12-31   577.157      94.656  6.097
rosbag2_2021_12_31-09_58_13      2  2021-12-31    96.062      85.090  1.129
rosbag2_2021_12_31-10_04_22      2  2021-12-31    99.344     100.703  0.987
rosbag2_2021_12_31-10_12_31      0  2021-12-31    98.431      83.251  1.182

Kept 95 missions | max kept ratio 0.339 | median offset 19.04 s | chunk duration 14.08 ms (median; v24's corrected-anchor formula) | non-modal chunks dropped 0 | missions with IMU timestamp inversions 95 (total 481924)
class       0  1  2  3  4
day                      
2021-12-22  2  2  2  2  2
2021-12-31  7  7  6  8  8
2022-01-04  3  2  4  4  4
2022-01-05  3  4  2  2  4
2022-01-10  3  4  4  4  2
[HOOK data] PASS — exclusion set == the 4 known names: ['rosbag2_2021_12_29-09_10_31', 'rosbag2_2021_12_31-09_58_13', 'rosbag2_2021_12_31-10_0

True

In [11]:
# ==============================================================================
# 3c. ONE SET OF PAIRED WINDOWS (v24 CORRECTED_ALIGNMENT): audio (64 x 80) and IMU (128 x 6, file order) from the SAME
#     real time span of the SAME mission. Every model below reads these arrays (unimodal models ignore one half).
# ==============================================================================
def paired_windows(name, P):
    s_times, a_times, dur, _, _ = timebase(P)
    tel, feats = P["imu_x"], P["feats"]
    max_time = min(s_times[-1], a_times[-1])
    out, i = [], 0
    t_start = max(0.0, float(a_times[0]))                  # v24: CORRECTED_WINDOW_START = "audio_onset"
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue
        s_idx = np.where(s_mask)[0]
        pick = s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]
        tau_s_abs = s_times[pick]
        x_s = tel[pick].astype(np.float32)                 # real file-order samples (IMU notebook rule)
        x_s_v24 = np.stack([np.interp(tau_s_abs, s_times[s_mask], tel[s_mask, c]) for c in range(N_SENSOR_CHANNELS)],
                           axis=1).astype(np.float32)      # v24's np.interp form, kept only for the comparison count
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            a_pick = a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]
            tau_a_abs = a_times[a_pick]
            x_a = feats[a_pick].astype(np.float32)         # == v24's np.interp at exact (strictly increasing) chunk times
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
            x_a = np.stack([np.interp(tau_a_abs, a_times, feats[:, c], left=0.0, right=0.0) for c in range(N_AUDIO_FEAT)],
                           axis=1).astype(np.float32)
        rel = (t_start + WINDOW_SEC / 2) / dur
        out.append({"window_id": f"{name}#{i:04d}", "mission": name, "cls": LABEL[name], "day": DAY[name],
                    "t_start": t_start, "t_end": t_end,
                    "phase": "start" if rel < STEADY[0] else ("end" if rel > STEADY[1] else "steady"),
                    "x_a": x_a, "x_s": x_s, "tau_a": ((tau_a_abs - t_start) / WINDOW_SEC).astype(np.float32),
                    "tau_s": ((tau_s_abs - t_start) / WINDOW_SEC).astype(np.float32),
                    "v24_interp_differs": bool(not np.array_equal(x_s, x_s_v24)),
                    "v24_interp_maxdiff": float(np.abs(x_s - x_s_v24).max())})
        i += 1
        t_start += STEP_SEC                                 # v24: accumulated float step
    return out


_W = []
for name in KEPT:
    _W.extend(paired_windows(name, PARSED[name]))
WMETA = pd.DataFrame([{k: v for k, v in w.items() if k not in ("x_a", "x_s", "tau_a", "tau_s")} for w in _W])
XA = np.stack([w["x_a"] for w in _W])                      # (N, 64, 80)
XS = np.stack([w["x_s"] for w in _W])                      # (N, 128, 6)
TAU_A = np.stack([w["tau_a"] for w in _W])                 # (N, 64)   in window units (v24 dataset: tau / window_sec)
TAU_S = np.stack([w["tau_s"] for w in _W])                 # (N, 128)
YW = WMETA.cls.values.astype(np.int64)
del _W
gc.collect()
print(f"Paired windows: {len(WMETA)} | XA {XA.shape} | XS {XS.shape} | per class {np.bincount(YW, minlength=N_CLASSES).tolist()} | "
      f"per phase {WMETA.phase.value_counts().to_dict()}")
print(f"IMU: windows where the file-order direct pick differs from v24's np.interp form: "
      f"{int(WMETA.v24_interp_differs.sum())} / {len(WMETA)} (max abs diff {WMETA.v24_interp_maxdiff.max():.4g}) — "
      "nonzero only where a window's IMU timestamps go backwards")

# ---- pairing hooks ----
_span_ok = bool((TAU_A >= -1e-6).all() and (TAU_A <= 1 + 1e-6).all() and (TAU_S >= -1e-6).all() and (TAU_S <= 1 + 1e-6).all())
hook("pairing", _span_ok and WMETA.window_id.is_unique,
     "every window's audio tokens AND IMU samples lie inside its own [t_start, t_start + 2.56 s] span of ONE mission "
     "(both come from a single record: same mission id, same span)")
_rng_h = np.random.default_rng(0)
_sample = np.sort(_rng_h.choice(len(WMETA), size=min(N_HASH_SAMPLE, len(WMETA)), replace=False))
_hash_ok = True
for j in _sample:
    r = WMETA.iloc[j]
    rebuilt = {w["window_id"]: w for w in paired_windows(r.mission, PARSED[r.mission])}[r.window_id]
    same = sha16(rebuilt["x_a"], rebuilt["x_s"]) == sha16(XA[j], XS[j]) and abs(rebuilt["t_start"] - r.t_start) < 1e-12
    _hash_ok &= same
hook("pairing", _hash_ok, f"hash check on {len(_sample)} sampled windows: stored (audio, IMU) == rebuilt from the raw "
                          f"mission at the recorded (mission, t_start)")
_fin = bool(np.isfinite(XA).all() and np.isfinite(XS).all())
hook("pairing", _fin and XA.shape[1:] == (TA_AUDIO, N_AUDIO_FEAT) and XS.shape[1:] == (TS_SENSOR, N_SENSOR_CHANNELS),
     f"shapes audio {XA.shape[1:]}, IMU {XS.shape[1:]}; all finite: {_fin}")
PAIRED_SHA = sha16(XA, XS)
print(f"Paired-window content sha1: {PAIRED_SHA}")

Paired windows: 5238 | XA (5238, 64, 80) | XS (5238, 128, 6) | per class [961, 1094, 974, 1118, 1091] | per phase {'steady': 4637, 'end': 601}
IMU: windows where the file-order direct pick differs from v24's np.interp form: 5238 / 5238 (max abs diff 84.07) — nonzero only where a window's IMU timestamps go backwards
[HOOK pairing] PASS — every window's audio tokens AND IMU samples lie inside its own [t_start, t_start + 2.56 s] span of ONE mission (both come from a single record: same mission id, same span)
[HOOK pairing] PASS — hash check on 25 sampled windows: stored (audio, IMU) == rebuilt from the raw mission at the recorded (mission, t_start)
[HOOK pairing] PASS — shapes audio (64, 80), IMU (128, 6); all finite: True
Paired-window content sha1: 44c04ca18ec7a131


In [12]:
# ==============================================================================
# 4. FOLDS (identical for all 6 models)
# ==============================================================================
def inner_val_fallback_2026(pool, k):
    """Standalone audio/IMU notebooks' rule: per class, ~15% (min 1) of the training missions -> validation, seed 2026."""
    rng = np.random.default_rng([INNER_VAL_FALLBACK_SEED, k])
    train, val = [], []
    for c in range(N_CLASSES):
        ms = sorted(m for m in pool if LABEL[m] == c)
        perm = rng.permutation(len(ms))
        n_val = max(1, int(round(INNER_VAL_FRACTION * len(ms))))
        val += [ms[i] for i in perm[:n_val]]
        train += [ms[i] for i in perm[n_val:]]
    return train, val


def inner_val_v27(pool, k):
    """v27/v28 `_split_inner_val(pool, protocol_idx=0, fold_idx=k)` (fingerprint 58e3c037c513ace2), seed 2027."""
    rng = np.random.default_rng([INNER_VAL_SEED_V27, 0, k])
    train, val = [], []
    for c in range(N_CLASSES):
        ms = sorted(m for m in pool if LABEL[m] == c)
        perm = rng.permutation(len(ms))
        n_val = max(1, int(round(INNER_VAL_FRACTION * len(ms))))
        assert len(ms) - n_val >= 1
        val += [ms[i] for i in perm[:n_val]]
        train += [ms[i] for i in perm[n_val:]]
    return train, val


inner_val = inner_val_fallback_2026 if INNER_VAL_RULE == "fallback_2026" else inner_val_v27
FOLDS = {}
for k, day in enumerate(DAYS):
    test = sorted(m for m in KEPT if DAY[m] == day)
    tr, va = inner_val([m for m in KEPT if DAY[m] != day], k)
    FOLDS[k] = {"day": day, "train": sorted(tr), "val": sorted(va), "test": test}
    assert not (set(tr) & set(va)) and not ((set(tr) | set(va)) & set(test)) and set(tr) | set(va) | set(test) == set(KEPT)
    assert day not in {DAY[m] for m in tr + va}
hook("folds", sorted(m for f in FOLDS.values() for m in f["test"]) == sorted(KEPT),
     "5 day-out test folds partition the 95 missions; no mission in two of train/val/test; test day never in training")

FOLD_PREP = {}
for k, f in FOLDS.items():
    idx = {s: np.where(WMETA.mission.isin(f[s]).values)[0] for s in ("train", "val", "test")}
    assert set(WMETA.mission.values[idx["train"]]) == set(f["train"])
    a_tr = XA[idx["train"]].reshape(-1, N_AUDIO_FEAT)
    s_tr = XS[idx["train"]].reshape(-1, N_SENSOR_CHANNELS)
    a_mu, a_sd = a_tr.mean(0, dtype=np.float64).astype(np.float32), (a_tr.std(0, dtype=np.float64) + AUDIO_STD_EPS).astype(np.float32)
    s_mu, s_sd = s_tr.mean(0, dtype=np.float64).astype(np.float32), (s_tr.std(0, dtype=np.float64) + SENSOR_STD_EPS).astype(np.float32)
    del a_tr, s_tr
    counts = np.bincount(YW[idx["train"]], minlength=N_CLASSES)
    FOLD_PREP[k] = {"idx": idx, "a_mu": a_mu, "a_sd": a_sd, "s_mu": s_mu, "s_sd": s_sd, "zhash": sha16(a_mu, a_sd, s_mu, s_sd),
                    "class_weights": (counts.sum() / (N_CLASSES * np.maximum(counts, 1))).astype(np.float32)}

print(f"Inner validation: {INNER_VAL_RULE}")
print(f"{'fold':<5}{'test day':<12}{'missions tr/val/test':<22}{'test missions/class':<22}{'windows tr/val/test':<22}{'z-score hash':<18}class weights")
for k, f in FOLDS.items():
    P = FOLD_PREP[k]
    print(f"{k:<5}{f['day']:<12}{'/'.join(str(len(f[s])) for s in ('train', 'val', 'test')):<22}"
          f"{str(np.bincount([LABEL[m] for m in f['test']], minlength=N_CLASSES).tolist()):<22}"
          f"{'/'.join(str(len(P['idx'][s])) for s in ('train', 'val', 'test')):<22}{P['zhash']:<18}"
          f"{np.round(P['class_weights'], 3).tolist()}" + ("  <- SMALL fold (10 missions)" if f["day"] == SMALL_FOLD_DAY else ""))

[HOOK folds] PASS — 5 day-out test folds partition the 95 missions; no mission in two of train/val/test; test day never in training
Inner validation: fallback_2026
fold test day    missions tr/val/test  test missions/class   windows tr/val/test   z-score hash      class weights
0    2021-12-22  72/13/10              [2, 2, 2, 2, 2]       3932/704/602          b5726f1a0d498224  [1.0440000295639038, 0.9919999837875366, 1.0460000038146973, 0.972000002861023, 0.953000009059906]  <- SMALL fold (10 missions)
1    2021-12-31  49/10/36              [7, 7, 6, 8, 8]       2700/564/1974         720c42d948aed91e  [1.1180000305175781, 0.9520000219345093, 0.9940000176429749, 0.9419999718666077, 1.0110000371932983]
2    2022-01-04  67/11/17              [3, 2, 4, 4, 4]       3680/621/937          dd723497b43897b8  [1.0709999799728394, 0.9330000281333923, 1.1360000371932983, 0.9359999895095825, 0.9559999704360962]
3    2022-01-05  69/11/15              [3, 4, 2, 2, 4]       3852/596/790          5359d

In [13]:
# ==============================================================================
# 0.5b  ASSERT the rebuilt windows and folds equal version 29's recorded values
# ==============================================================================
V29_PAIRED_SHA, V29_N_WINDOWS = "44c04ca18ec7a131", 5238
V29_FOLD = {0: ("b5726f1a0d498224", (3932, 704, 602)), 1: ("720c42d948aed91e", (2700, 564, 1974)),
            2: ("dd723497b43897b8", (3680, 621, 937)), 3: ("5359d0c5720fd958", (3852, 596, 790)),
            4: ("a9b3c1d3fd502d2b", (3678, 625, 935))}
hook("v29-identity", PAIRED_SHA == V29_PAIRED_SHA and len(WMETA) == V29_N_WINDOWS,
     f"paired-window content sha1 {PAIRED_SHA} == v29's {V29_PAIRED_SHA}; {len(WMETA)} windows")
hook("v29-identity", all(FOLD_PREP[k]["zhash"] == V29_FOLD[k][0] and
                         tuple(len(FOLD_PREP[k]["idx"][s]) for s in ("train", "val", "test")) == V29_FOLD[k][1] for k in range(len(DAYS))),
     "every fold's z-score hash and train/val/test window counts == v29's (same 95 missions, folds, inner validation seed 2026)")

[HOOK v29-identity] PASS — paired-window content sha1 44c04ca18ec7a131 == v29's 44c04ca18ec7a131; 5238 windows
[HOOK v29-identity] PASS — every fold's z-score hash and train/val/test window counts == v29's (same 95 missions, folds, inner validation seed 2026)


True

### 0.6 Models — v29's cells, verbatim
- **D:** v24's classes, verbatim.
- **A and I:** v28's `UnimodalMamba`.
- **New encoders:** the audio sigmoid gate (80 parameters) and the IMU `Conv1d(6→6, k7)`.
- **F0, F1, F2:** `GatedFusionMamba`.

The parameter table and the real-Mamba hook are v29's cell, run on this notebook's model list.

In [14]:
# ==============================================================================
# 5a. CONDITION D — v24's model + loss classes, VERBATIM (v22 == v24 == v27); build_fusion_blocks: real mamba_ssm only
# ==============================================================================
import math


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    """v24's function without the CPU fallback branch: real mamba_ssm.Mamba blocks only."""
    return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])


# ---- VERBATIM from version 24/minor-project-v24.ipynb (model cell + AeroSyncLoss from the training cell) ----
# sha1[:10] of each source (identical in v22, v24, v27): ContinuousTimeEmbedding 6eb422a2ea, HeterogeneousTemporalTokenization f378aaef09, GridLayerNorm 3a64dba834, PairwiseLACA da33fc4f61, BiModalLACAFusion 567cf3d5d6, MambaBlock b88a5bf138, AeroSyncMamba 9d843e21a5, verify_architecture a7c78390a9, AeroSyncLoss 468046c494

class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding [+ time embedding(tau)]); modality 0 = audio, 1 = sensor.
    v22: time_enabled = HTT_TIME_EMBED_ENABLED. When False the time embedding is skipped entirely (no temporal
    position information enters through HTT); the module is still built so every condition draws the same RNG."""

    def __init__(self, d_model: int, time_enabled: bool = True):
        super().__init__()
        self.d_model = d_model
        self.time_enabled = time_enabled
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        if not self.time_enabled:
            return self.norm(z + e_m)                       # HTT_TIME_EMBED_ENABLED = False
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)   # beta = 1.0 (lag bias ON) or exactly 0.0 (OFF)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, htt_time: bool = True, laca_lag: bool = False):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.htt_time_enabled = htt_time                  # HTT_TIME_EMBED_ENABLED
        self.laca_lag_enabled = laca_lag                  # LACA_LAG_BIAS_ENABLED
        self.laca_beta = LACA_BETA if laca_lag else 0.0   # OFF = exactly 0.0 (v18's beta = 0 code path)

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model, time_enabled=htt_time)
        self.laca = BiModalLACAFusion(d_model, beta=self.laca_beta)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model):
    """Finalized architecture, read from the constructed modules, including both v22 switches."""
    assert model.htt.time_enabled == model.htt_time_enabled
    beta = LACA_BETA if model.laca_lag_enabled else 0.0
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != {beta} for LACA_LAG_BIAS_ENABLED={model.laca_lag_enabled}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}

In [15]:
# ==============================================================================
# 5b. A / I (v28 UnimodalMamba) and the NEW encoders + F0 / F1 / F2
# ==============================================================================
class UnimodalMamba(nn.Module):
    """v28's single-modality path: encoder -> + modality embedding -> LayerNorm -> 2 Mamba blocks -> attention pooling
    -> classifier. Reads only its own modality from the shared paired windows."""

    def __init__(self, modality):
        super().__init__()
        self.modality = modality
        if modality == "audio":
            self.audio_encoder = nn.Sequential(nn.Linear(N_AUDIO_FEAT, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(),
                                               nn.Linear(D_MODEL, D_MODEL))
        else:
            self.sensor_encoder = nn.Sequential(
                nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
                nn.Conv1d(64, D_MODEL, kernel_size=3, padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())
        self.modality_embedding = nn.Embedding(1, D_MODEL)
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = build_fusion_blocks(D_MODEL, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))

    def forward(self, xa, xs, ta, ts, gate_override=None):
        z = self.audio_encoder(xa) if self.modality == "audio" else self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        h = self.token_norm(z + self.modality_embedding.weight.view(1, 1, -1))
        for block in self.mamba_blocks:
            h = block(h)
        w = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier((h * w).sum(dim=1)), None, None, {}


class GatedAudioEncoder(nn.Module):
    """NEW audio encoder: ONE learnable elementwise feature mask m = sigmoid(w), w in R^80 (a fixed learned mask, NOT
    input-dependent), then the existing audio projection."""

    def __init__(self):
        super().__init__()
        self.gate_logit = nn.Parameter(torch.zeros(N_AUDIO_FEAT))      # m = 0.5 at init; zeros_ draws no RNG
        self.proj = nn.Sequential(nn.Linear(N_AUDIO_FEAT, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(), nn.Linear(D_MODEL, D_MODEL))

    def mask(self):
        return torch.sigmoid(self.gate_logit)

    def forward(self, xa):
        return self.proj(self.mask() * xa)


class ConvIMUEncoder(nn.Module):
    """NEW IMU encoder: ONE Conv1d(6 -> 6, k=7, 'same' padding) before the existing IMU projection (unchanged)."""

    def __init__(self):
        super().__init__()
        self.pre_conv = nn.Conv1d(N_SENSOR_CHANNELS, IMU_PRECONV_CHANNELS, kernel_size=IMU_PRECONV_KERNEL,
                                  padding=IMU_PRECONV_KERNEL // 2)
        self.proj = nn.Sequential(
            nn.Conv1d(IMU_PRECONV_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, D_MODEL, kernel_size=3, padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())

    def forward(self, xs):                                  # (B, T, 6) -> (B, T, D)
        return self.proj(self.pre_conv(xs.transpose(1, 2))).transpose(1, 2)


class GatedFusionMamba(nn.Module):
    """F0: [A ; I] (no gate). F1: softmax gate [g_a, g_i]. F2: two independent sigmoid gates.
    A_summary / I_summary (mean pooling, as D) feed the gate AND the InfoNCE / KL losses (no second pooled vector)."""

    def __init__(self, kind):
        super().__init__()
        assert kind in ("F0", "F1", "F2")
        self.kind = kind
        self.audio_encoder = GatedAudioEncoder()
        self.sensor_encoder = ConvIMUEncoder()
        self.modality_embeddings = nn.Embedding(2, D_MODEL)          # D's HTT with the time embedding OFF
        self.token_norm = nn.LayerNorm(D_MODEL)
        if kind == "F1":
            self.gate_mlp = nn.Sequential(nn.Linear(2 * D_MODEL, GATE_HIDDEN), nn.GELU(), nn.Linear(GATE_HIDDEN, 2))
        elif kind == "F2":
            self.gate_a = nn.Sequential(nn.Linear(2 * D_MODEL, GATE_HIDDEN), nn.GELU(), nn.Linear(GATE_HIDDEN, 1))
            self.gate_i = nn.Sequential(nn.Linear(2 * D_MODEL, GATE_HIDDEN), nn.GELU(), nn.Linear(GATE_HIDDEN, 1))
        self.mamba_blocks = build_fusion_blocks(D_MODEL, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))
        self.aux_a = nn.Linear(D_MODEL, N_CLASSES)
        self.aux_s = nn.Linear(D_MODEL, N_CLASSES)

    def forward(self, xa, xs, ta, ts, gate_override=None):
        B = xa.shape[0]
        e = self.modality_embeddings.weight
        A = self.token_norm(self.audio_encoder(xa) + e[0].view(1, 1, -1))     # (B, 64, D)
        I = self.token_norm(self.sensor_encoder(xs) + e[1].view(1, 1, -1))    # (B, 128, D)
        a_sum, i_sum = A.mean(dim=1), I.mean(dim=1)
        s = torch.cat([a_sum, i_sum], dim=-1)
        if self.kind == "F0":
            g_a = g_i = torch.ones(B, device=xa.device)
        elif self.kind == "F1":
            g = F.softmax(self.gate_mlp(s), dim=-1)
            g_a, g_i = g[:, 0], g[:, 1]
        else:
            g_a, g_i = torch.sigmoid(self.gate_a(s)).squeeze(-1), torch.sigmoid(self.gate_i(s)).squeeze(-1)
        if gate_override is not None:                                          # diagnostics only: constant gates
            g_a = torch.full((B,), float(gate_override[0]), device=xa.device)
            g_i = torch.full((B,), float(gate_override[1]), device=xa.device)
        H = torch.cat([g_a.view(B, 1, 1) * A, g_i.view(B, 1, 1) * I], dim=1)   # (B, 192, D)
        for block in self.mamba_blocks:
            H = block(H)
        w = F.softmax(self.pool_attn(H), dim=1)
        logits = self.classifier((H * w).sum(dim=1))
        return (logits, {"z_a": a_sum, "z_s": i_sum}, {"y_a": self.aux_a(a_sum), "y_s": self.aux_s(i_sum)},
                {"g_a": g_a, "g_i": g_i})


class DWrapper(nn.Module):
    """Condition D with the common interface; the wrapped AeroSyncMamba is v24's class, unmodified."""

    def __init__(self):
        super().__init__()
        self.net = AeroSyncMamba(htt_time=HTT_TIME_EMBED_ENABLED, laca_lag=LACA_LAG_BIAS_ENABLED)
        self.mamba_blocks = self.net.mamba_blocks

    def forward(self, xa, xs, ta, ts, gate_override=None):
        logits, pooled, aux, lags = self.net(xa, xs, ta, ts)
        return logits, pooled, aux, {}


def build_model(name):
    if name == "A":
        return UnimodalMamba("audio")
    if name == "I":
        return UnimodalMamba("sensor")
    if name == "D":
        return DWrapper()
    return GatedFusionMamba(name)

In [16]:
# ==============================================================================
# 5c. PARAMETER COUNTS, ONE-LINE ARCHITECTURE SUMMARY, identical real-Mamba config (HOOK models)
# ==============================================================================
ARCH = {
    "A": "audio 80 -> MLP(256) -> +e_mod, LN -> 2x Mamba -> attn pool -> MLP head | CE",
    "I": "IMU 6ch -> CNN(k5,k3 -> 256) -> +e_mod, LN -> 2x Mamba -> attn pool -> MLP head | CE",
    "D": "v24 AeroSyncMamba: MLP audio + CNN IMU -> HTT(time OFF) -> LACA x-attn (4 heads, lag x 0) -> [Ha;SEP;Hs] -> 2x Mamba | CE+InfoNCE+KL",
    "F0": "gate(80) audio + Conv1d(k7) IMU -> HTT-off tokens -> [A;I] -> 2x Mamba | CE+InfoNCE+KL",
    "F1": "as F0 + softmax gate MLP(512->64->2) scaling [g_a*A ; g_i*I] | CE+InfoNCE+KL",
    "F2": "as F0 + two sigmoid gate MLPs(512->64->1) scaling [g_a*A ; g_i*I] | CE+InfoNCE+KL",
}
N_PARAMS, _cfg = {}, {}
seed_all(42)
for name in MODELS:
    m = build_model(name)
    N_PARAMS[name] = sum(p.numel() for p in m.parameters())
    blocks = list(m.mamba_blocks)
    _cfg[name] = (tuple(type(b.mamba) for b in blocks), tuple((b.mamba.d_state, b.mamba.d_conv, b.mamba.expand) for b in blocks),
                  sum(p.numel() for p in m.mamba_blocks.parameters()))
    extra = ""
    if name in NEW_MODELS:
        extra = (f" | added: audio gate {m.audio_encoder.gate_logit.numel()} params, IMU pre-Conv1d "
                 f"{sum(p.numel() for p in m.sensor_encoder.pre_conv.parameters())} params"
                 + (f", gate MLP {sum(p.numel() for p in m.gate_mlp.parameters()):,}" if name == "F1" else "")
                 + (f", gate MLPs {sum(p.numel() for n_, p in m.named_parameters() if n_.startswith(('gate_a', 'gate_i'))):,}" if name == "F2" else ""))
    print(f"  {name:<3} {N_PARAMS[name]:>11,} params | {ARCH[name]}{extra}")
    if name == "D":
        verify_architecture(m.net)
        assert not m.net.htt.time_enabled and m.net.laca.laca_as.beta == 0.0 and m.net.laca.laca_sa.beta == 0.0
        print(f"      D check: HTT time embedding {'ON' if m.net.htt.time_enabled else 'OFF'} | LACA beta "
              f"{m.net.laca.laca_as.beta}/{m.net.laca.laca_sa.beta} | LACA heads {m.net.laca.laca_as.num_heads} (Q/K/V + "
              f"softmax cross-attention ACTIVE) | gamma_as/gamma_sa init {float(m.net.laca.gamma_as):.1f}/{float(m.net.laca.gamma_sa):.1f}")
    del m
_ref = _cfg[MODELS[0]]
hook("models", all(all(t is MAMBA_CLASS for t in c[0]) and c[1] == ((MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND),) * MAMBA_NUM_BLOCKS
                   and c[2] == EXPECTED_MAMBA_PARAMS for c in _cfg.values()) and MAMBA_CLASS.__module__.startswith("mamba_ssm"),
     f"all {len(MODELS)} models use {MAMBA_NUM_BLOCKS} x {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} (d_state "
     f"{MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}, expand {MAMBA_EXPAND}; {EXPECTED_MAMBA_PARAMS:,} Mamba params each)")
torch.cuda.empty_cache()

  A     1,012,358 params | audio 80 -> MLP(256) -> +e_mod, LN -> 2x Mamba -> attn pool -> MLP head | CE
  I       977,350 params | IMU 6ch -> CNN(k5,k3 -> 256) -> +e_mod, LN -> 2x Mamba -> attn pool -> MLP head | CE
  F1    1,100,516 params | as F0 + softmax gate MLP(512->64->2) scaling [g_a*A ; g_i*I] | CE+InfoNCE+KL | added: audio gate 80 params, IMU pre-Conv1d 258 params, gate MLP 32,962
[HOOK models] PASS — all 3 models use 2 x mamba_ssm.modules.mamba_simple.Mamba (d_state 16, d_conv 2, expand 2; 874,496 Mamba params each)


In [17]:
# ==============================================================================
# 0.6b  PORTED SETTINGS (printed for checking against version 29)
# ==============================================================================
print("Ported from version 29 (values as defined in its cells above):")
print(f"  Mamba: {MAMBA_NUM_BLOCKS} blocks x {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__}, d_state {MAMBA_D_STATE}, d_conv "
      f"{MAMBA_D_CONV}, expand {MAMBA_EXPAND}, {EXPECTED_MAMBA_PARAMS:,} Mamba params | d_model {D_MODEL} | dropout {DROPOUT}")
print(f"  Encoders: audio {N_AUDIO_FEAT} features x Ta {TA_AUDIO} (sigmoid gate in F-models), IMU {N_SENSOR_CHANNELS} ch x "
      f"Ts {TS_SENSOR} file order (Conv1d k{IMU_PRECONV_KERNEL} in F-models) | F1 gate MLP hidden {GATE_HIDDEN}")
print(f"  Windows {WINDOW_SEC} s / stride {STEP_SEC} s | inner validation {INNER_VAL_RULE} ({INNER_VAL_FRACTION:.0%} per class)")
print(f"  Loss: A, I = class-weighted CE only; F1 (and D/F0/F2) = class-weighted CE + {LAMBDA_ALIGN} InfoNCE (T "
      f"{INFONCE_TEMPERATURE}) + {LAMBDA_CONS} KL, from the mean-pooled A_summary / I_summary")
print(f"  Optimization: AdamW lr {LEARNING_RATE}, wd {WEIGHT_DECAY}, cosine to {LR_MIN}, clip {GRAD_CLIP_NORM}, {EPOCHS} epochs, "
      f"batch {BATCH_SIZE}, best-validation-macro-F1 checkpoint; seeds {SEEDS}")
print("  Parameters: " + ", ".join(f"{m} {N_PARAMS[m]:,}" for m in MODELS))
print("  Determinism: established in v29 (mamba_ssm CUDA backward jitter only; 0.00 effect on validation macro-F1); not re-run.")

Ported from version 29 (values as defined in its cells above):
  Mamba: 2 blocks x mamba_ssm.modules.mamba_simple.Mamba, d_state 16, d_conv 2, expand 2, 874,496 Mamba params | d_model 256 | dropout 0.15
  Encoders: audio 80 features x Ta 64 (sigmoid gate in F-models), IMU 6 ch x Ts 128 file order (Conv1d k7 in F-models) | F1 gate MLP hidden 64
  Windows 2.56 s / stride 1.28 s | inner validation fallback_2026 (15% per class)
  Loss: A, I = class-weighted CE only; F1 (and D/F0/F2) = class-weighted CE + 0.1 InfoNCE (T 0.07) + 0.1 KL, from the mean-pooled A_summary / I_summary
  Optimization: AdamW lr 0.0001, wd 0.0001, cosine to 1e-06, clip 1.0, 20 epochs, batch 32, best-validation-macro-F1 checkpoint; seeds [42, 123]
  Parameters: A 1,012,358, I 977,350, F1 1,100,516
  Determinism: established in v29 (mamba_ssm CUDA backward jitter only; 0.00 effect on validation macro-F1); not re-run.


### 0.7 Training — v29's `train_run` and resumable sweep, verbatim
- **Order:** fold-major, then model (A, I, F1), then seed.
- **Logging:** each run is appended to `v32_expert_training_results.csv` immediately. Rows already there, with their
  `.npz` file present, are skipped.
- **Per run:** `seed_all(seed)`, a fresh model, a seeded DataLoader generator, and GPU memory freed afterwards.
- **Progress:** the `==> N/30 complete | ... | ETA` lines are v29's.

In [18]:
# ==============================================================================
# 6. METRICS, PREDICTION, train_run
# ==============================================================================
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score)
from scipy import stats

LABELS = list(range(N_CLASSES))


def compute_metrics(y, P):
    pred = P.argmax(1)
    try:
        auroc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=LABELS))
    except ValueError:
        auroc = float("nan")
    return {"accuracy": float(accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=LABELS, zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)), "macro_auroc": auroc}


def fold_arrays(k, split):
    P_, idx = FOLD_PREP[k], FOLD_PREP[k]["idx"][split]
    return (((XA[idx] - P_["a_mu"]) / P_["a_sd"]).astype(np.float32), ((XS[idx] - P_["s_mu"]) / P_["s_sd"]).astype(np.float32),
            TAU_A[idx], TAU_S[idx], YW[idx])


def make_loader(arrs, shuffle, seed):
    ds = TensorDataset(*[torch.from_numpy(np.ascontiguousarray(a)) for a in arrs])
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True,
                      worker_init_fn=worker_init_fn, generator=make_generator(seed) if shuffle else None)


@torch.no_grad()
def predict(model, loader, gate_override=None):
    model.eval()
    probs, ga, gi = [], [], []
    for xa, xs, ta, ts, _ in loader:
        logits, _, _, ex = model(xa.to(device), xs.to(device), ta.to(device), ts.to(device), gate_override=gate_override)
        probs.append(F.softmax(logits, dim=-1).float().cpu().numpy())
        if "g_a" in ex:
            ga.append(ex["g_a"].float().cpu().numpy())
            gi.append(ex["g_i"].float().cpu().numpy())
    return np.concatenate(probs), (np.concatenate(ga) if ga else None), (np.concatenate(gi) if gi else None)


def mission_level(missions, y, P):
    df = pd.DataFrame(P, columns=LABELS)
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g["y"].first().values, g[LABELS].mean().values, g["y"].first().index.values


def train_run(name, k, seed, epochs=EPOCHS, save=True, verbose=True, record_steps=False):
    t0 = time.time()
    seed_all(seed)                                           # at the START of every run
    model = build_model(name).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()[:12]
    assert all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks)
    cw = torch.tensor(FOLD_PREP[k]["class_weights"], device=device)
    criterion = AeroSyncLoss(cw) if name in FUSED_MODELS else nn.CrossEntropyLoss(weight=cw)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    tr = make_loader(fold_arrays(k, "train"), True, seed)
    va = make_loader(fold_arrays(k, "val"), False, seed)
    te_arrs = fold_arrays(k, "test")
    te = make_loader(te_arrs, False, seed)
    tag = f"[{name} fold {k} ({FOLDS[k]['day']}) seed {seed}]"
    if verbose:
        print(f"{tag} init {init_sha1} | {N_PARAMS[name]:,} params | windows train {len(tr.dataset)} / val "
              f"{len(va.dataset)} / test {len(te.dataset)} | z-score {FOLD_PREP[k]['zhash']}")
    best_f1, best_state, best_epoch, history, step_losses = 0.0, None, None, [], []
    for epoch in range(1, epochs + 1):
        model.train()
        run = defaultdict(float)
        nb = 0
        for xa, xs, ta, ts, y in tr:
            xa, xs, ta, ts, y = (t.to(device, non_blocking=True) for t in (xa, xs, ta, ts, y))
            optimizer.zero_grad()
            logits, pooled, aux, _ = model(xa, xs, ta, ts)
            if name in FUSED_MODELS:
                loss, parts = criterion(logits, y, pooled, aux)
            else:
                loss, parts = criterion(logits, y), {}
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            run["loss"] += loss.item()
            for kk, v in parts.items():
                run[kk] += v
            nb += 1
            if record_steps:
                step_losses.append(loss.item())
        scheduler.step()
        assert math.isfinite(run["loss"]), f"{tag} non-finite loss"
        vP, _, _ = predict(model, va)
        vf1 = compute_metrics(va.dataset.tensors[4].numpy(), vP)["macro_f1"]
        history.append((run["loss"] / nb, vf1))
        improved = vf1 > best_f1
        if improved:
            best_f1, best_state, best_epoch = vf1, copy.deepcopy(model.state_dict()), epoch
        if verbose:
            extra = f" (cls {run['loss_cls'] / nb:.4f}, align {run['loss_align'] / nb:.4f}, cons {run['loss_cons'] / nb:.4f})" if parts else ""
            print(f"{tag} epoch {epoch:02d}/{epochs} | train loss {run['loss'] / nb:.4f}{extra} | val macro-F1 {vf1:.4f}"
                  + ("  * best" if improved else ""))
    if best_state is None:
        best_state, best_epoch = copy.deepcopy(model.state_dict()), epochs
    model.load_state_dict(best_state)
    res = {"model": name, "fold": k, "fold_day": FOLDS[k]["day"], "seed": seed, "init_sha1": init_sha1, "history": history,
           "step_losses": step_losses, "best_epoch": best_epoch, "best_val_macro_f1": round(best_f1, 6),
           "zstats_sha1": FOLD_PREP[k]["zhash"], "n_params": N_PARAMS[name]}
    if save:
        P, ga, gi = predict(model, te)
        y = te_arrs[4]
        meta = WMETA.iloc[FOLD_PREP[k]["idx"]["test"]]
        my, mP, _ = mission_level(meta.mission.values, y, P)
        res.update({f"win_{kk}": v for kk, v in compute_metrics(y, P).items()})
        res.update({f"mis_{kk}": v for kk, v in compute_metrics(my, mP).items()})
        extra = {}
        if name in ("F1", "F2"):
            # content-adaptivity effect: replace each window's gates by their test-set mean, compare predictions
            P_const, _, _ = predict(model, te, gate_override=(float(ga.mean()), float(gi.mean())))
            res.update({"g_a_mean": float(ga.mean()), "g_a_std": float(ga.std()), "g_a_min": float(ga.min()), "g_a_max": float(ga.max()),
                        "g_i_mean": float(gi.mean()), "g_i_std": float(gi.std()), "g_i_min": float(gi.min()), "g_i_max": float(gi.max()),
                        "both_ambiguous_frac": float(((ga < 0.5) & (gi < 0.5)).mean()),
                        "gate_effect_mean_abs_dp": float(np.abs(P - P_const).mean()),
                        "gate_effect_argmax_changed": float((P.argmax(1) != P_const.argmax(1)).mean())})
            extra.update({"g_a": ga.astype(np.float32), "g_i": gi.astype(np.float32)})
            print(f"{tag} GATES (test): g_a mean {ga.mean():.3f} sd {ga.std():.3f} [{ga.min():.3f}, {ga.max():.3f}] | g_i mean "
                  f"{gi.mean():.3f} sd {gi.std():.3f} [{gi.min():.3f}, {gi.max():.3f}] | "
                  f"{'VARIES' if max(ga.std(), gi.std()) > GATE_STD_THRESHOLD else 'NEAR-CONSTANT'} (sd threshold "
                  f"{GATE_STD_THRESHOLD}) | both<0.5: {res['both_ambiguous_frac']:.3f} | constant-gate effect: mean|dp| "
                  f"{res['gate_effect_mean_abs_dp']:.4f}, argmax changed {res['gate_effect_argmax_changed']:.3f}")
        if name in NEW_MODELS:
            mask = model.audio_encoder.mask().detach().float().cpu().numpy()
            extra["audio_mask"] = mask.astype(np.float32)
            if k == 0 and seed == SEEDS[0]:
                top = np.argsort(-mask)[:10]
                print(f"{tag} learned audio mask (80 = 5 ch x bins 1-16): top 10 " + ", ".join(
                    f"ch{j // 16 + 1}/bin{j % 16 + 1}={mask[j]:.3f}" for j in top) + f" | min {mask.min():.3f} max {mask.max():.3f}")
        fname = f"{name}_{k}_{seed}"
        np.savez(PROBS_DIR / f"{fname}.npz", window_id=meta.window_id.values.astype(str), mission=meta.mission.values.astype(str),
                 day=meta.day.values.astype(str), cls=y, phase=meta.phase.values.astype(str), probs=P.astype(np.float32), **extra)
        torch.save(best_state, CKPT_DIR / f"{fname}.pt")
        res.update({"class_weights": json.dumps(np.round(FOLD_PREP[k]["class_weights"], 4).tolist()),
                    "n_train_windows": len(tr.dataset), "n_val_windows": len(va.dataset), "n_test_windows": len(te.dataset),
                    "probs_file": str(PROBS_DIR / f"{fname}.npz"), "checkpoint": str(CKPT_DIR / f"{fname}.pt")})
    res["train_seconds"] = round(time.time() - t0, 1)
    del model, optimizer, scheduler, best_state, tr, va, te
    gc.collect()
    torch.cuda.empty_cache()
    return res

In [19]:
# ==============================================================================
# 8. SWEEP
# ==============================================================================
CSV_FIELDS = ["model", "fold", "fold_day", "seed", "n_train_windows", "n_val_windows", "n_test_windows",
              "win_accuracy", "win_macro_f1", "win_balanced_accuracy", "win_macro_auroc",
              "mis_accuracy", "mis_macro_f1", "mis_balanced_accuracy", "mis_macro_auroc",
              "best_epoch", "best_val_macro_f1", "init_sha1", "zstats_sha1", "class_weights", "n_params", "train_seconds",
              "g_a_mean", "g_a_std", "g_a_min", "g_a_max", "g_i_mean", "g_i_std", "g_i_min", "g_i_max",
              "both_ambiguous_frac", "gate_effect_mean_abs_dp", "gate_effect_argmax_changed",
              "probs_file", "checkpoint", "finished_at"]


def load_done():
    if not RESULTS_CSV.exists():
        return {}
    df = pd.read_csv(RESULTS_CSV)
    return {(r.model, int(r.fold), int(r.seed)): r for r in df.itertuples()
            if (PROBS_DIR / f"{r.model}_{int(r.fold)}_{int(r.seed)}.npz").exists()}


def append_row(res):
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({**res, "finished_at": datetime.now().isoformat(timespec="seconds")})


PLAN = [(m, k, s) for k in range(len(DAYS)) for m in MODELS for s in SEEDS]
_done = load_done()
print(f"{len(PLAN)} runs planned ({len(MODELS)} models x {len(DAYS)} folds x {len(SEEDS)} seeds) | already logged "
      f"{sum(p in _done for p in PLAN)}")
_t0, _n_new, SWEEP_T0 = time.time(), 0, time.time()
for i, (m, k, s) in enumerate(PLAN, 1):
    _done = load_done()
    if (m, k, s) in _done:
        continue
    if m == "F2" and ("F1", k, s) not in _done:
        print(f"[skip] F2 fold {k} seed {s}: F1 for this fold/seed is not logged yet (F2 runs only after F1)")
        continue
    res = train_run(m, k, s)
    append_row(res)
    _n_new += 1
    n_complete = len([p for p in PLAN if p in load_done()])
    eta = (len(PLAN) - n_complete) * (time.time() - _t0) / _n_new / 60
    print(f"==> {n_complete}/{len(PLAN)} complete | {m} fold {k} seed {s}: window macro-F1 {res['win_macro_f1']:.4f}, "
          f"mission macro-F1 {res['mis_macro_f1']:.4f}, best epoch {res['best_epoch']}, {res['train_seconds']:.0f} s | "
          f"ETA {eta:.0f} min")
SWEEP_MINUTES = (time.time() - SWEEP_T0) / 60
print(f"Sweep finished: {len([p for p in PLAN if p in load_done()])}/{len(PLAN)} runs logged | this session trained "
      f"{_n_new} runs in {SWEEP_MINUTES:.1f} min")

30 runs planned (3 models x 5 folds x 2 seeds) | already logged 0
[A fold 0 (2021-12-22) seed 42] init a2901c1c4c22 | 1,012,358 params | windows train 3932 / val 704 / test 602 | z-score b5726f1a0d498224
[A fold 0 (2021-12-22) seed 42] epoch 01/20 | train loss 0.8839 | val macro-F1 0.7240  * best
[A fold 0 (2021-12-22) seed 42] epoch 02/20 | train loss 0.3671 | val macro-F1 0.8072  * best
[A fold 0 (2021-12-22) seed 42] epoch 03/20 | train loss 0.2305 | val macro-F1 0.8593  * best
[A fold 0 (2021-12-22) seed 42] epoch 04/20 | train loss 0.1959 | val macro-F1 0.8274
[A fold 0 (2021-12-22) seed 42] epoch 05/20 | train loss 0.1531 | val macro-F1 0.8332
[A fold 0 (2021-12-22) seed 42] epoch 06/20 | train loss 0.1274 | val macro-F1 0.8334
[A fold 0 (2021-12-22) seed 42] epoch 07/20 | train loss 0.1001 | val macro-F1 0.8192
[A fold 0 (2021-12-22) seed 42] epoch 08/20 | train loss 0.1086 | val macro-F1 0.8181
[A fold 0 (2021-12-22) seed 42] epoch 09/20 | train loss 0.0949 | val macro-F1 0.852

In [20]:
# ==============================================================================
# 0.8  ALL EXPERT .npz FILES PRESENT + PLAUSIBILITY CHECK vs version 29's prior numbers (not a gate)
# ==============================================================================
_expected = [(m, k, s) for m in MODELS for k in range(len(DAYS)) for s in SEEDS]
_absent = [f"{m}_{k}_{s}.npz" for m, k, s in _expected if not (PROBS_DIR / f"{m}_{k}_{s}.npz").exists()]
hook("section0", not _absent, f"all {len(_expected)} expert runs completed and their .npz files exist in {PROBS_DIR}"
                              + (f" — MISSING {_absent}" if _absent else ""))

[HOOK section0] PASS — all 30 expert runs completed and their .npz files exist in /kaggle/working/v32_npz


True

In [21]:
# ==============================================================================
# v29 helper functions, VERBATIM from version 29/minor-project-v29.ipynb (cell 24): prf_from_cm, cluster_bootstrap_f1, level_block
# ==============================================================================



def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def cluster_bootstrap_f1(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Macro-F1 over B resamples of MISSIONS within class (never windows)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    pos = {m: i for i, m in enumerate(ms)}
    cnt, mcls = np.zeros((len(ms), N_CLASSES), int), np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        cms[:, c, :] = cnt[rows[rng.integers(0, len(rows), (B, len(rows)))]].sum(1)
    return np.array([prf_from_cm(cm)[2].mean() for cm in cms])


def level_block(missions, y, P):
    pred = P.argmax(1)
    m = compute_metrics(y, P)
    cm = confusion_matrix(y, pred, labels=LABELS)
    _, rec, _ = prf_from_cm(cm)
    lo, hi = np.percentile(cluster_bootstrap_f1(missions, y, pred), [2.5, 97.5])
    err = pred != y
    return {"accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "f1_ci_low": lo, "f1_ci_high": hi,
            "balanced_accuracy": m["balanced_accuracy"], "macro_auroc": m["macro_auroc"],
            **{f"recall_c{c}": rec[c] for c in range(N_CLASSES)},
            "faulty_pred_normal": float((pred[y > 0] == 0).mean()), "normal_pred_faulty": float((pred[y == 0] > 0).mean()),
            "adjacent_error_share": float((np.abs(pred - y)[err] == 1).mean()) if err.any() else float("nan"), "cm": cm}

In [22]:
# ==============================================================================
# 0.8b  POOLED DAY-OUT RESULTS OF THE FRESHLY TRAINED EXPERTS + DELTA vs v29 (plausibility only)
# ==============================================================================
_rows = []
for m in MODELS:
    per_seed = []
    for s in SEEDS:
        parts = [dict(np.load(PROBS_DIR / f"{m}_{k}_{s}.npz")) for k in range(len(DAYS))]
        ids = np.concatenate([p["window_id"] for p in parts])
        o = np.argsort(ids)
        per_seed.append((ids[o], np.concatenate([p["probs"] for p in parts])[o]))
    ids = per_seed[0][0]
    meta = WMETA.set_index("window_id").loc[ids].reset_index()
    Pm = np.mean([p for _, p in per_seed], axis=0)
    wb = level_block(meta.mission.values, meta.cls.values, Pm)
    my, mP, mn = mission_level(meta.mission.values, meta.cls.values, Pm)
    mb = level_block(mn, my, mP)
    lim = PLAUSIBILITY_FACTOR * NOISE_FLOOR_F1
    dw, dm = wb["macro_f1"] - PRIOR_V29[m][0], mb["macro_f1"] - PRIOR_V29[m][1]
    _rows.append({"model": m, "window macro-F1 [95% CI]": f"{wb['macro_f1']:.3f} [{wb['f1_ci_low']:.3f}, {wb['f1_ci_high']:.3f}]",
                  "mission macro-F1 [95% CI]": f"{mb['macro_f1']:.3f} [{mb['f1_ci_low']:.3f}, {mb['f1_ci_high']:.3f}]",
                  "v29 prior (window / mission)": f"{PRIOR_V29[m][0]:.3f} / {PRIOR_V29[m][1]:.3f}",
                  "delta window": round(dw, 4), "delta mission": round(dm, 4),
                  "plausibility": "OK" if max(abs(dw), abs(dm)) <= lim else f"UNUSUALLY LARGE (> {lim:.3f})"})
SECTION0_TABLE = pd.DataFrame(_rows).set_index("model")
print(f"Freshly trained experts — pooled day-out (5 folds x seeds {SEEDS}, seed-averaged probabilities):")
print(SECTION0_TABLE.to_string())
print(f"(Deltas vs version 29's earlier run are a plausibility check, NOT a reproduction gate: a different training run "
      f"differs by roughly the {NOISE_FLOOR_F1} seed-to-seed noise floor; flagged only above {PLAUSIBILITY_FACTOR} x that.)")
if not TRAIN_EXTRA_MODELS:
    print("\nHARDCODED REFERENCE VALUES, NOT REPRODUCED IN THIS SESSION, from a prior run (v29, pooled day-out window "
          "macro-F1; context only, never used in any computation): D 0.871 | F0 0.838 | F2 0.853")
_csv = pd.read_csv(RESULTS_CSV)
print(f"\nSection 0 finished: {len(_csv)} runs logged in {RESULTS_CSV.name}; logged training time "
      f"{_csv.train_seconds.sum() / 60:.1f} min; section wall time {(time.time() - T0_SECTION0) / 60:.1f} min.")
SECTION0_HOOKS = dict(HOOKS)                                # kept for the final hook table (Part 1 re-creates HOOKS)

Freshly trained experts — pooled day-out (5 folds x seeds [42, 123], seed-averaged probabilities):
      window macro-F1 [95% CI] mission macro-F1 [95% CI] v29 prior (window / mission)  delta window  delta mission plausibility
model                                                                                                                          
A         0.801 [0.745, 0.853]      0.864 [0.793, 0.927]                0.795 / 0.864        0.0062        -0.0004           OK
I         0.893 [0.865, 0.917]      0.968 [0.934, 1.000]                0.893 / 0.968        0.0001        -0.0001           OK
F1        0.866 [0.817, 0.911]      0.894 [0.832, 0.948]                0.869 / 0.894       -0.0026         0.0003           OK
(Deltas vs version 29's earlier run are a plausibility check, NOT a reproduction gate: a different training run differs by roughly the 0.016 seed-to-seed noise floor; flagged only above 3 x that.)

HARDCODED REFERENCE VALUES, NOT REPRODUCED IN THIS SESSION, fro

# Parts 0 and 1 — three-way complementarity and the decision router (built on Section 0's expert probabilities)

**Inputs.**
- **Probabilities:** the per-window day-out test probabilities that **Section 0 of this notebook just trained and
  saved** in `/kaggle/working/v32_npz/`. `version 29/Npz/` is used only if Section 0's files are absent and a complete
  set is present there.
- **CSVs:** `version 29/Results/` and `version 30/Results/`, for the parameter-count context.

**The experts are trained in this notebook**, in Section 0, from version 29's code. From here on they are fixed:
Parts 0 and 1 use only their saved probabilities, and the only model trained in Parts 0–1 is the small router MLP.

**Missing files stop the notebook.** If a required file is missing, the exact expected paths are printed and the
notebook stops there. Nothing is substituted.

**Experts:**
- **A:** audio-only;
- **I:** IMU-only;
- **F1:** the v29 fusion model (new encoders + content-adaptive softmax gate).

All three share v29's protocol: 95 missions, 5238 paired windows, 5 leave-one-day-out folds, seeds 42 and 123.

**Schema** (checked in section 1, not assumed): `window_id`, `mission`, `day`, `cls`, `phase` and `probs` (N × 5,
post-softmax). **There are no logits.** Calibration therefore uses the probability-based power-law temperature
method, which is **the method, not a fallback**:

$$p_{cal} \propto p_{raw}^{1/T}$$

**Where the router's training data comes from.** v29's training code, run in Section 0, saves **test-fold
probabilities only**: each file covers only its own test day. There are no inner-validation probabilities, so the
specified split (halves of each fold's inner-validation missions) is impossible. Section 1 checks this from the files
themselves.

**The substitute used here.** For outer fold k, `CAL_HALF` and `ROUTER_HALF` are carved out of the **other four days'
saved test predictions**. It is a mission-level split, stratified by class, with seed 7. The router is evaluated
**only on day k**, which it never sees in any form. This keeps the outer test day completely held out.

Splitting day k's own missions instead would leave only part of day k to evaluate on, and would put day-k labels into
calibration and router training.

> **⚠️ Caveats of this substitute** (restated with every result):
> 1. **Model mismatch.** The windows used for calibration and router training carry predictions from the *other*
>    folds' expert checkpoints, not fold k's. The router learns how the experts typically behave on an unseen day and
>    applies that to fold k's checkpoints.
> 2. **Weak indirect leakage.** Those other folds' experts were trained on data that included day k. Day k's labels
>    therefore influence the router's training features through the experts' weights, though never directly.
> 3. **Not the leakage you specified.** Checkpoint selection used each fold's inner-validation missions, and none of
>    the windows used here are inner-validation windows. The router is still **not** a clean nested cross-validation.
>    That would need an extra, nested level of expert retraining, which is not implemented here (section 9).

## 1. Setup, file listing, schema check, data availability

In [23]:
# ==============================================================================
# 1a. SETUP — locate the three permitted folders and print their exact contents
# ==============================================================================
import glob, hashlib, itertools, json, math, os, random, re, subprocess, sys, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize_scalar

T0_NOTEBOOK = time.time()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "v32_outputs"
WORK.mkdir(parents=True, exist_ok=True)
PERMITTED = ["version 29/Npz", "version 29/Results", "version 30/Results"]


def _has_all(root):
    return all((root / p).is_dir() for p in PERMITTED)


# project root: the local checkout this notebook sits in (version 32/..), else a fresh clone of the public repo
ROOT = next((r for r in (Path.cwd().parent, Path.cwd()) if _has_all(r)), None)
if ROOT is None:
    ROOT = WORK / "AeroSync-Mamba"
    if not (ROOT / ".git").exists():
        print("Cloning the public project repo (--depth 1) to read the permitted folders ...")
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Sainava/AeroSync-Mamba.git", str(ROOT)], check=True)
print(f"Project root: {ROOT}")
LISTING = {}
for p in PERMITTED:
    d = ROOT / p
    LISTING[p] = sorted(f.name for f in d.iterdir()) if d.is_dir() else None
    print(f"\n{p}/ " + ("<< FOLDER NOT FOUND >>" if LISTING[p] is None else f"({len(LISTING[p])} files)"))
    for n in LISTING[p] or []:
        print(f"    {n}")
# .npz source: Section 0's output FIRST; version 29/Npz only if Section 0's set is incomplete and v29's is complete
V32_NPZ_DIR = WORK / "v32_npz"
_REQ_NAMES = [f"{m}_{k}_{s}.npz" for m in ("A", "I", "F1") for k in range(5) for s in (42, 123)]


def _complete(d):
    return d.is_dir() and all((d / f).exists() for f in _REQ_NAMES)


NPZ_DIR = next((d for d in (V32_NPZ_DIR, ROOT / "version 29" / "Npz") if _complete(d)), V32_NPZ_DIR)
print(f"\nPer-window probabilities read from: {NPZ_DIR} "
      + ("(Section 0's freshly trained experts)" if NPZ_DIR == V32_NPZ_DIR else "(FALLBACK: version 29/Npz — Section 0's set is incomplete)"))
print(f"{NPZ_DIR}/ ({len(list(NPZ_DIR.glob('*.npz'))) if NPZ_DIR.is_dir() else 0} .npz files): "
      + ", ".join(sorted(f.name for f in NPZ_DIR.glob("*.npz"))) if NPZ_DIR.is_dir() else "<< NOT FOUND >>")
V29_RUNS_CSV = ROOT / "version 29" / "Results" / "gated_fusion_results.csv"
V30_RUNS_CSV = ROOT / "version 30" / "Results" / "crossattn_results.csv"

# ---- constants ----
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
SMALL_FOLD_DAY = "2021-12-22"
SEEDS = [42, 123]
EXPERTS = ["A", "I", "F1"]
N_CLASSES = 5
EXPECTED_N_WINDOWS, EXPECTED_N_MISSIONS = 5238, 95
SPLIT_SEED = 7
T_GRID = np.unique(np.concatenate([np.exp(np.linspace(np.log(0.05), np.log(20.0), 400)), [1.0]]))
ROUTER_HIDDEN = 32
ROUTER_EPOCHS = 500           # full-batch Adam, fixed a priori (nothing is selected on a test day)
ROUTER_LR = 1e-2
ROUTER_WD = 1e-4
TUNED_GRID_STEP = 0.05
BOOTSTRAP_RESAMPLES, BOOTSTRAP_SEED = 2000, 2028
NOISE_FLOOR_F1 = 0.016
COLLAPSE_SD = 0.05
HOOKS = defaultdict(list)


def hook(name, ok, msg, stop=False):
    ok = bool(ok)
    HOOKS[name].append((ok, msg))
    print(f"[HOOK {name}] {'PASS' if ok else 'FAIL'} — {msg}")
    if not ok and stop:
        raise RuntimeError(f"HOOK {name} FAILED: {msg}")
    return ok


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


hook("listing", all(v is not None for v in LISTING.values()), "all three permitted folders found and listed above")

Project root: /kaggle/working/AeroSync-Mamba

version 29/Npz/ (10 files)
    I_0_123.npz
    I_0_42.npz
    I_1_123.npz
    I_1_42.npz
    I_2_123 (1).npz
    I_2_123.npz
    I_2_42.npz
    I_3_42.npz
    I_4_123.npz
    I_4_42.npz

version 29/Results/ (3 files)
    gated_fusion_results.csv
    v29_results_table.csv
    v29_summary.txt

version 30/Results/ (3 files)
    crossattn_results.csv
    v30_results_table.csv
    v30_summary.txt

Per-window probabilities read from: /kaggle/working/v32_npz (Section 0's freshly trained experts)
/kaggle/working/v32_npz/ (30 .npz files): A_0_123.npz, A_0_42.npz, A_1_123.npz, A_1_42.npz, A_2_123.npz, A_2_42.npz, A_3_123.npz, A_3_42.npz, A_4_123.npz, A_4_42.npz, F1_0_123.npz, F1_0_42.npz, F1_1_123.npz, F1_1_42.npz, F1_2_123.npz, F1_2_42.npz, F1_3_123.npz, F1_3_42.npz, F1_4_123.npz, F1_4_42.npz, I_0_123.npz, I_0_42.npz, I_1_123.npz, I_1_42.npz, I_2_123.npz, I_2_42.npz, I_3_123.npz, I_3_42.npz, I_4_123.npz, I_4_42.npz
[HOOK listing] PASS — all three pe

True

In [24]:
# ==============================================================================
# 1b. SCHEMA — read from an actual file, not assumed
# ==============================================================================
_NPZ_RE = re.compile(r"^(A|I|D|F0|F1|F2)_(\d)_(\d+)\.npz$")
FILES = {}
_NPZ_NAMES = sorted(f.name for f in NPZ_DIR.glob("*.npz")) if NPZ_DIR.is_dir() else []
for n in _NPZ_NAMES:
    m_ = _NPZ_RE.match(n)
    if m_:
        FILES[(m_.group(1), int(m_.group(2)), int(m_.group(3)))] = NPZ_DIR / n
IGNORED = [n for n in _NPZ_NAMES if not _NPZ_RE.match(n)]
if IGNORED:
    print(f"Files not matching {{model}}_{{fold}}_{{seed}}.npz, ignored: {IGNORED}")
if not FILES:
    loud(f"No per-window probability files in {NPZ_DIR}. Nothing in this notebook can run.")
    raise RuntimeError("no .npz files")
_first = sorted(FILES)[0]
_z = np.load(FILES[_first])
SCHEMA = {k: (tuple(_z[k].shape), str(_z[k].dtype)) for k in _z.files}
print(f"Schema of {FILES[_first].name}:")
for k, (sh, dt) in SCHEMA.items():
    print(f"    {k:<10} shape {sh}  dtype {dt}")
EXPECTED_KEYS = {"window_id", "mission", "day", "cls", "probs"}
HAS_LOGITS = any("logit" in k.lower() for k in SCHEMA)
print(f"Expected keys present: {EXPECTED_KEYS <= set(SCHEMA)} | extra keys: {sorted(set(SCHEMA) - EXPECTED_KEYS)} | "
      f"logits present: {HAS_LOGITS}")
print("-> Calibration method: " + ("logits are present, but the probability-based power-law method is still used for all "
                                   "experts, for consistency" if HAS_LOGITS else
                                   "probability-based power-law temperature p^(1/T) — THE method (no logits were saved)"))
hook("schema", EXPECTED_KEYS <= set(SCHEMA) and SCHEMA["probs"][0][1:] == (N_CLASSES,),
     f"keys {sorted(SCHEMA)}; probs is (N, {N_CLASSES})", stop=True)

Schema of A_0_42.npz:
    window_id  shape (602,)  dtype <U32
    mission    shape (602,)  dtype <U27
    day        shape (602,)  dtype <U10
    cls        shape (602,)  dtype int64
    phase      shape (602,)  dtype <U6
    probs      shape (602, 5)  dtype float32
Expected keys present: True | extra keys: ['phase'] | logits present: False
-> Calibration method: probability-based power-law temperature p^(1/T) — THE method (no logits were saved)
[HOOK schema] PASS — keys ['cls', 'day', 'mission', 'phase', 'probs', 'window_id']; probs is (N, 5)


True

In [25]:
# ==============================================================================
# 1c. DATA AVAILABILITY — every (expert, fold, seed) file must exist; stop otherwise
# ==============================================================================
REQUIRED = [(m, k, s) for m in EXPERTS for k in range(len(DAYS)) for s in SEEDS]
MISSING = [key for key in REQUIRED if key not in FILES]
print(f"Required: {len(REQUIRED)} files ({EXPERTS} x folds 0-4 x seeds {SEEDS}); present {len(REQUIRED) - len(MISSING)}")
for m in EXPERTS:
    print(f"    {m:<3} {sum((m, k, s) in FILES for k in range(len(DAYS)) for s in SEEDS)}/10")
print(f"Other models' files present (not used): {sorted({k[0] for k in FILES} - set(EXPERTS))}")
if MISSING:
    loud("REQUIRED FILES MISSING — Part 0 and Part 1 cannot run. Expected paths:\n"
         + "\n".join(f"   {NPZ_DIR / f'{m}_{k}_{s}.npz'}" for m, k, s in MISSING)
         + "\nThese are produced by Section 0 of this notebook — check its run log (v32_expert_training_results.csv) for "
           "runs that did not finish, re-run Section 0 (it resumes), then re-run from here. Nothing is substituted for them.")
    raise RuntimeError(f"{len(MISSING)} required .npz files missing — stopping (see the list above).")

# ---- load, align, and detect whether any file holds inner-validation windows ----
RAW, TEST_ONLY = {}, True
for m, k, s in REQUIRED:
    z = np.load(FILES[(m, k, s)])
    d = {key: z[key] for key in ("window_id", "mission", "day", "cls", "probs")}
    other = d["day"] != DAYS[k]
    if other.any():
        TEST_ONLY = False
    RAW[(m, k, s)] = d
print(f"\nFiles contain ONLY their own fold's test-day windows: {TEST_ONLY}")
INNER_VAL_AVAILABLE = not TEST_ONLY
print("-> " + ("Inner-validation windows ARE present in the files: CAL_HALF / ROUTER_HALF come from each fold's inner-validation "
               "missions, as specified." if INNER_VAL_AVAILABLE else
               "NO inner-validation probabilities exist (v29 saved test folds only). FALLBACK IN USE: for outer fold k, "
               "CAL_HALF / ROUTER_HALF are carved out of the OTHER four days' saved test predictions; day k is never used "
               "for calibration or router training (see the caveats in the header)."))


def test_part(d, k):
    sel = d["day"] == DAYS[k]
    return {key: v[sel] for key, v in d.items()}


# master window order: all test windows, sorted by id; every expert/seed must match it exactly
_ids = np.concatenate([test_part(RAW[("I", k, SEEDS[0])], k)["window_id"] for k in range(len(DAYS))])
ORDER_IDS = np.sort(_ids)
P = {}                                        # P[(expert, seed)] -> (N, 5) raw probabilities aligned to ORDER_IDS
META = None
for m in EXPERTS:
    for s in SEEDS:
        parts = [test_part(RAW[(m, k, s)], k) for k in range(len(DAYS))]
        cat = {key: np.concatenate([p[key] for p in parts]) for key in parts[0]}
        o = np.argsort(cat["window_id"])
        cat = {key: v[o] for key, v in cat.items()}
        assert np.array_equal(cat["window_id"], ORDER_IDS), f"{m} seed {s}: window ids differ from I's"
        pr = np.clip(cat["probs"].astype(np.float64), 1e-12, None)
        P[(m, s)] = pr / pr.sum(1, keepdims=True)
        if META is None:
            META = pd.DataFrame({key: cat[key] for key in ("window_id", "mission", "day", "cls")})
            META["fold"] = META.day.map({d: i for i, d in enumerate(DAYS)})
        else:
            assert np.array_equal(META.mission.values, cat["mission"]) and np.array_equal(META.cls.values, cat["cls"])
Y = META.cls.values.astype(int)
IDS_SHA = hashlib.sha1("\n".join(ORDER_IDS).encode()).hexdigest()[:16]
hook("windows", len(ORDER_IDS) == EXPECTED_N_WINDOWS and META.mission.nunique() == EXPECTED_N_MISSIONS,
     f"A, I, F1 x seeds {SEEDS} cover the IDENTICAL {len(ORDER_IDS)} test windows / {META.mission.nunique()} missions "
     f"(window-id sha1 {IDS_SHA}, same for every expert and seed; same missions and classes)", stop=True)

Required: 30 files (['A', 'I', 'F1'] x folds 0-4 x seeds [42, 123]); present 30
    A   10/10
    I   10/10
    F1  10/10
Other models' files present (not used): []

Files contain ONLY their own fold's test-day windows: True
-> NO inner-validation probabilities exist (v29 saved test folds only). FALLBACK IN USE: for outer fold k, CAL_HALF / ROUTER_HALF are carved out of the OTHER four days' saved test predictions; day k is never used for calibration or router training (see the caveats in the header).
[HOOK windows] PASS — A, I, F1 x seeds [42, 123] cover the IDENTICAL 5238 test windows / 95 missions (window-id sha1 1d23df9ff89e449b, same for every expert and seed; same missions and classes)


True

In [26]:
# ==============================================================================
# v29 helper functions, VERBATIM from version 29/minor-project-v29.ipynb (cell 18): compute_metrics, mission_level
# ==============================================================================
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score)
from scipy import stats
LABELS = list(range(N_CLASSES))


def compute_metrics(y, P):
    pred = P.argmax(1)
    try:
        auroc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=LABELS))
    except ValueError:
        auroc = float("nan")
    return {"accuracy": float(accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=LABELS, zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)), "macro_auroc": auroc}


def mission_level(missions, y, P):
    df = pd.DataFrame(P, columns=LABELS)
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g["y"].first().values, g[LABELS].mean().values, g["y"].first().index.values

In [27]:
# ==============================================================================
# v29 helper functions, VERBATIM from version 29/minor-project-v29.ipynb (cell 24): prf_from_cm, cluster_bootstrap_f1, level_block
# ==============================================================================



def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def cluster_bootstrap_f1(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Macro-F1 over B resamples of MISSIONS within class (never windows)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    pos = {m: i for i, m in enumerate(ms)}
    cnt, mcls = np.zeros((len(ms), N_CLASSES), int), np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        cms[:, c, :] = cnt[rows[rng.integers(0, len(rows), (B, len(rows)))]].sum(1)
    return np.array([prf_from_cm(cm)[2].mean() for cm in cms])


def level_block(missions, y, P):
    pred = P.argmax(1)
    m = compute_metrics(y, P)
    cm = confusion_matrix(y, pred, labels=LABELS)
    _, rec, _ = prf_from_cm(cm)
    lo, hi = np.percentile(cluster_bootstrap_f1(missions, y, pred), [2.5, 97.5])
    err = pred != y
    return {"accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "f1_ci_low": lo, "f1_ci_high": hi,
            "balanced_accuracy": m["balanced_accuracy"], "macro_auroc": m["macro_auroc"],
            **{f"recall_c{c}": rec[c] for c in range(N_CLASSES)},
            "faulty_pred_normal": float((pred[y > 0] == 0).mean()), "normal_pred_faulty": float((pred[y == 0] > 0).mean()),
            "adjacent_error_share": float((np.abs(pred - y)[err] == 1).mean()) if err.any() else float("nan"), "cm": cm}

## 2. PART 0 — Three-way complementarity (no training)
The inputs are A's, I's and F1's saved day-out test probabilities, averaged over seeds 42 and 123 per window. The mission
prediction is the argmax of the mean window probability. Everything is reported at window **and** mission level:
- **8-way breakdown:** which of the three experts are right on each window or mission.
- **Oracle ceilings:** 2-expert (A + I) and 3-expert (A + I + F1). A window counts as correct if any of the included
  experts is correct. For macro-F1, when all included experts are wrong, the oracle predicts the equal-average argmax.
- **Error redundancy:** P(F1 wrong | I wrong) vs P(F1 wrong | I right), and the same for A vs I and A vs F1. If the
  first is much larger than the second, the two experts fail on the same cases.

This part is context for reading Part 1. It does not gate it.

In [28]:
# ==============================================================================
# 2. PART 0 — THREE-WAY COMPLEMENTARITY
# ==============================================================================
PAVG = {m: np.mean([P[(m, s)] for s in SEEDS], axis=0) for m in EXPERTS}


def level_preds(level):
    """Per-expert predictions and truth at window or mission level (seed-averaged probabilities)."""
    if level == "window":
        return {m: PAVG[m].argmax(1) for m in EXPERTS}, Y, {m: PAVG[m] for m in EXPERTS}
    out, probs = {}, {}
    for m in EXPERTS:
        ym, pm, names = mission_level(META.mission.values, Y, PAVG[m])
        out[m], probs[m] = pm.argmax(1), pm
    return out, ym, probs


def oracle_pred(preds, y, probs, members):
    any_ok = np.any([preds[m] == y for m in members], axis=0)
    fallback = np.mean([probs[m] for m in members], axis=0).argmax(1)
    return np.where(any_ok, y, fallback), any_ok


PART0 = {}
for level in ("window", "mission"):
    preds, y, probs = level_preds(level)
    ok = {m: preds[m] == y for m in EXPERTS}
    n = len(y)
    print(f"\n=== PART 0 — {level.upper()} level ({n} {level}s) ===")
    rows = []
    for combo in itertools.product([True, False], repeat=3):
        sel = np.all([ok[m] == c for m, c in zip(EXPERTS, combo)], axis=0)
        right = [m for m, c in zip(EXPERTS, combo) if c]
        label = ("all 3 right" if len(right) == 3 else "all 3 wrong" if not right else
                 f"exactly 2 right: {'+'.join(right)}" if len(right) == 2 else f"exactly 1 right: {right[0]} only")
        rows.append({"pattern": label, "count": int(sel.sum()), "share": sel.mean()})
    tab = pd.DataFrame(rows).set_index("pattern")
    print(tab.round(4).to_string())
    res = {"breakdown": tab}
    for name, members in (("A+I", ["A", "I"]), ("A+I+F1", ["A", "I", "F1"])):
        op, any_ok = oracle_pred(preds, y, probs, members)
        res[name] = {"accuracy": any_ok.mean(), "macro_f1": f1_score(y, op, average="macro", labels=LABELS, zero_division=0)}
    for m in EXPERTS:
        res[m] = {"accuracy": ok[m].mean(), "macro_f1": f1_score(y, preds[m], average="macro", labels=LABELS, zero_division=0)}
    print("accuracy / macro-F1: " + " | ".join(f"{k} {res[k]['accuracy']:.4f} / {res[k]['macro_f1']:.4f}"
                                               for k in ("A", "I", "F1", "A+I", "A+I+F1")))
    print(f"ORACLE CEILING: 2-expert (A+I) accuracy {res['A+I']['accuracy']:.4f}, macro-F1 {res['A+I']['macro_f1']:.4f}  vs  "
          f"3-expert (A+I+F1) accuracy {res['A+I+F1']['accuracy']:.4f}, macro-F1 {res['A+I+F1']['macro_f1']:.4f}  "
          f"(F1 adds {res['A+I+F1']['accuracy'] - res['A+I']['accuracy']:+.4f} accuracy)")
    cond = {}
    for x, z in (("F1", "I"), ("A", "I"), ("A", "F1")):
        pw = (~ok[x] & ~ok[z]).sum() / max((~ok[z]).sum(), 1)
        pr = (~ok[x] & ok[z]).sum() / max(ok[z].sum(), 1)
        cond[(x, z)] = (pw, pr)
        print(f"P({x} wrong | {z} wrong) = {pw:.3f} (n {int((~ok[z]).sum())})   vs   P({x} wrong | {z} right) = {pr:.3f} "
              f"(n {int(ok[z].sum())})   ratio {pw / pr if pr > 0 else float('inf'):.1f}x")
    res["cond"] = cond
    PART0[level] = res

_w = PART0["window"]
_gain = _w["A+I+F1"]["accuracy"] - _w["A+I"]["accuracy"]
_red = _w["cond"][("F1", "I")]
PART0_STATEMENT = (
    f"Window level: the A+I oracle already reaches {_w['A+I']['accuracy']:.3f} accuracy; adding F1 raises the ceiling by "
    f"{_gain:+.3f} (to {_w['A+I+F1']['accuracy']:.3f}). F1's errors are "
    + ("largely REDUNDANT with I's" if _red[0] > 3 * max(_red[1], 1e-9) else "only partly redundant with I's")
    + f" (P(F1 wrong | I wrong) = {_red[0]:.2f} vs P(F1 wrong | I right) = {_red[1]:.2f}). "
    + ("F1 adds little headroom beyond A+I as a third expert (< 0.01), so any router gain must come mostly from choosing "
       "between A and I." if _gain < 0.01 else
       "F1 adds some headroom as a third expert, so a router can in principle gain from including it.")
    + f" Mission level: A+I oracle {PART0['mission']['A+I']['accuracy']:.3f} vs A+I+F1 {PART0['mission']['A+I+F1']['accuracy']:.3f}.")
print("\nPART 0 STATEMENT: " + PART0_STATEMENT)
print("How to read Part 1: the oracle is an upper bound that requires knowing the answer; a router can only approach it "
      "to the extent that the experts' own confidence signals which one is right.")


=== PART 0 — WINDOW level (5238 windows) ===
                          count   share
pattern                                
all 3 right                3846  0.7342
exactly 2 right: A+I         24  0.0046
exactly 2 right: A+F1       281  0.0536
exactly 1 right: A only      31  0.0059
exactly 2 right: I+F1       371  0.0708
exactly 1 right: I only     443  0.0846
exactly 1 right: F1 only     39  0.0074
all 3 wrong                 203  0.0388
accuracy / macro-F1: A 0.7984 / 0.8012 | I 0.8942 / 0.8931 | F1 0.8662 / 0.8664 | A+I 0.9538 / 0.9551 | A+I+F1 0.9612 / 0.9612
ORACLE CEILING: 2-expert (A+I) accuracy 0.9538, macro-F1 0.9551  vs  3-expert (A+I+F1) accuracy 0.9612, macro-F1 0.9612  (F1 adds +0.0074 accuracy)
P(F1 wrong | I wrong) = 0.422 (n 554)   vs   P(F1 wrong | I right) = 0.100 (n 4684)   ratio 4.2x
P(A wrong | I wrong) = 0.437 (n 554)   vs   P(A wrong | I right) = 0.174 (n 4684)   ratio 2.5x
P(A wrong | F1 wrong) = 0.922 (n 701)   vs   P(A wrong | F1 right) = 0.090 (n 4537)   r

## 3. PART 1 — CAL_HALF / ROUTER_HALF per outer fold
For outer fold k, the pool of windows comes from one of two sources:
- **If inner-validation windows exist** in the files: fold k's inner-validation windows.
- **Otherwise:** the fallback reported in section 1, the other four days' saved test windows.

The pool's missions are split into two halves: mission-level, stratified by class, with `default_rng([7, k])`. For
each class, the sorted missions are permuted and the first ⌊n/2⌋ go to CAL_HALF. The same split is used for both
seeds. **The router is evaluated only on day k's windows.**

In [29]:
# ==============================================================================
# 3. SPLITS
# ==============================================================================
SPLITS = {}
for k, day in enumerate(DAYS):
    if INNER_VAL_AVAILABLE:
        raise NotImplementedError("inner-validation windows were detected; wire them in here (not the case for v29's files)")
    pool = META[META.day != day]
    cal, rou = [], []
    rng = np.random.default_rng([SPLIT_SEED, k])
    for c in range(N_CLASSES):
        ms = sorted(pool[pool.cls == c].mission.unique())
        perm = rng.permutation(len(ms))
        h = len(ms) // 2
        cal += [ms[i] for i in perm[:h]]
        rou += [ms[i] for i in perm[h:]]
    test_idx = np.where(META.day.values == day)[0]
    cal_idx = np.where(META.mission.isin(cal).values)[0]
    rou_idx = np.where(META.mission.isin(rou).values)[0]
    SPLITS[k] = {"cal": cal_idx, "router": rou_idx, "test": test_idx, "cal_m": set(cal), "rou_m": set(rou)}
    disjoint = not (set(cal) & set(rou)) and not ((set(cal) | set(rou)) & set(META.mission.values[test_idx]))
    hook("splits", disjoint and day not in set(META.day.values[cal_idx]) | set(META.day.values[rou_idx]),
         f"fold {k} ({day}{', SMALL' if day == SMALL_FOLD_DAY else ''}): CAL_HALF {len(cal)} missions / {len(cal_idx)} windows, "
         f"ROUTER_HALF {len(rou)} missions / {len(rou_idx)} windows, test {len(set(META.mission.values[test_idx]))} "
         f"missions / {len(test_idx)} windows — disjoint; day {day} never in CAL/ROUTER; source = "
         + ("inner-validation" if INNER_VAL_AVAILABLE else "FALLBACK: other four days' saved test windows"))
SPLIT_SOURCE = "inner-validation windows" if INNER_VAL_AVAILABLE else "FALLBACK: other four days' saved test windows"

[HOOK splits] PASS — fold 0 (2021-12-22, SMALL): CAL_HALF 42 missions / 2245 windows, ROUTER_HALF 43 missions / 2391 windows, test 10 missions / 602 windows — disjoint; day 2021-12-22 never in CAL/ROUTER; source = FALLBACK: other four days' saved test windows
[HOOK splits] PASS — fold 1 (2021-12-31): CAL_HALF 29 missions / 1603 windows, ROUTER_HALF 30 missions / 1661 windows, test 36 missions / 1974 windows — disjoint; day 2021-12-31 never in CAL/ROUTER; source = FALLBACK: other four days' saved test windows
[HOOK splits] PASS — fold 2 (2022-01-04): CAL_HALF 38 missions / 2109 windows, ROUTER_HALF 40 missions / 2192 windows, test 17 missions / 937 windows — disjoint; day 2022-01-04 never in CAL/ROUTER; source = FALLBACK: other four days' saved test windows
[HOOK splits] PASS — fold 3 (2022-01-05): CAL_HALF 39 missions / 2168 windows, ROUTER_HALF 41 missions / 2280 windows, test 15 missions / 790 windows — disjoint; day 2022-01-05 never in CAL/ROUTER; source = FALLBACK: other four days'

## 4. Calibration — the probability-based power-law temperature
Each expert is calibrated independently per (fold, seed):

$$p_{cal} = \frac{p^{1/T}}{\sum p^{1/T}}$$

- **Fitting T:** T > 0 minimizes NLL on CAL_HALF. It is chosen by a 401-point log grid on [0.05, 20] that includes
  T = 1 exactly, then refined with a bounded 1-D search. Because T = 1 is a candidate, the NLL on CAL_HALF can never
  increase.
- **Argmax is unchanged.** A per-window power transform keeps the class order, so a single expert's window-level
  accuracy and macro-F1 do not change. Calibration matters for the **mixtures** (EQUAL, TUNED, router) and for
  **mission-level averaging**.

In [30]:
# ==============================================================================
# 4. CALIBRATION
# ==============================================================================
def calibrate(p, T):
    lp = np.log(p) / T
    lp -= lp.max(1, keepdims=True)
    q = np.exp(lp)
    return q / q.sum(1, keepdims=True)


def nll(p, y):
    return float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, None))))


def fit_T(p, y):
    grid = [nll(calibrate(p, T), y) for T in T_GRID]
    i = int(np.argmin(grid))
    lo, hi = np.log(T_GRID[max(i - 1, 0)]), np.log(T_GRID[min(i + 1, len(T_GRID) - 1)])
    r = minimize_scalar(lambda lt: nll(calibrate(p, math.exp(lt)), y), bounds=(lo, hi), method="bounded")
    return (math.exp(r.x), r.fun) if r.fun < grid[i] else (float(T_GRID[i]), grid[i])


CAL_T, CAL_ROWS = {}, []
for k in range(len(DAYS)):
    ci = SPLITS[k]["cal"]
    for s in SEEDS:
        for m in EXPERTS:
            before = nll(P[(m, s)][ci], Y[ci])
            T, after = fit_T(P[(m, s)][ci], Y[ci])
            CAL_T[(m, k, s)] = T
            CAL_ROWS.append({"fold": k, "seed": s, "expert": m, "T": T, "NLL before": before, "NLL after": after,
                             "reduced": after <= before + 1e-12})
CAL_TAB = pd.DataFrame(CAL_ROWS)
print("Fitted temperatures (T > 1 softens over-confident experts, T < 1 sharpens) and CAL_HALF NLL before -> after:")
print(CAL_TAB.pivot_table(index=["fold", "seed"], columns="expert", values=["T", "NLL before", "NLL after"]).round(4).to_string())
_bad = CAL_TAB[~CAL_TAB.reduced]
hook("calibration", _bad.empty, "calibration did not increase CAL_HALF NLL for any expert/fold/seed (T = 1 is always a "
                                "candidate, so it cannot)" + ("" if _bad.empty else f" — exceptions: {_bad.to_dict('records')}"))
_eq = CAL_TAB[CAL_TAB["NLL after"] >= CAL_TAB["NLL before"] - 1e-9]
if len(_eq):
    print(f"{len(_eq)} case(s) where calibration left NLL unchanged (best T ~ 1: that expert was already calibrated on CAL_HALF): "
          + ", ".join(f"{r.expert} fold {r.fold} seed {r.seed} (T {r.T:.3f})" for r in _eq.itertuples()))
Q = {(m, k, s): calibrate(P[(m, s)], CAL_T[(m, k, s)]) for m in EXPERTS for k in range(len(DAYS)) for s in SEEDS}
# Q[(m, k, s)] = expert m's seed-s probabilities over ALL windows, calibrated with fold k's temperature (used on fold k's CAL/ROUTER/test rows)

Fitted temperatures (T > 1 softens over-confident experts, T < 1 sharpens) and CAL_HALF NLL before -> after:
          NLL after                 NLL before                       T                
expert            A      F1       I          A      F1       I       A      F1       I
fold seed                                                                             
0    42      0.4921  0.3205  0.3686     0.9603  0.6500  0.5065  3.4750  3.3141  2.2974
     123     0.4993  0.3758  0.2471     1.1047  1.0993  0.2597  3.9625  4.4756  1.3225
1    42      0.4441  0.3249  0.3731     0.9516  0.7659  0.4649  3.7335  3.8376  2.0079
     123     0.5060  0.4364  0.3252     1.2319  1.2337  0.3687  4.2672  4.5147  1.5794
2    42      0.3003  0.1655  0.3819     0.4012  0.2177  0.5487  2.1712  1.9755  2.4630
     123     0.3204  0.1751  0.2359     0.4260  0.2790  0.2496  2.1222  2.4982  1.3591
3    42      0.5828  0.4327  0.3947     1.3895  1.0241  0.6244  4.3936  4.0218  2.8687
     123     0.6414  

## 5. Router — features, architecture, training
**Features per window** (calibrated with fold k's temperatures), 27 in total:

| group | dimensions |
|---|---|
| each expert's calibrated probability vector | 3 × 5 = 15 |
| each expert's entropy ÷ log 5 | 3 |
| each expert's max probability | 3 |
| each expert pair (A-I, A-F1, I-F1): L1 distance between the probability vectors, and whether the argmaxes agree | 3 × 2 = 6 |

The features are z-scored with ROUTER_HALF statistics.

**MLP:** `Linear(27 → 32) → ReLU → Linear(32 → 3)`, then `[w_A, w_I, w_F] = softmax(·)`.

**Prediction:** `p(y|x) = w_A·p_A + w_I·p_I + w_F·p_F`. It is trained with cross-entropy on that mixture (−log p(y|x)).

**Training setup:**
- It trains on ROUTER_HALF only: full-batch Adam, lr 1e-2, weight decay 1e-4, **500 epochs fixed in advance**.
  Nothing is selected on the test day, and there is no early stopping.
- The router seed s uses the experts' seed-s outputs, so each seed is an independent replicate of experts plus
  router.
- Seeding covers `random`, NumPy and torch (CPU + CUDA). Training is full-batch, so there is no DataLoader and no
  shuffle order to seed.
- In Parts 0–1, `torch` is used **only** for this 995-parameter router, on CPU. The experts were trained in
  Section 0 and are not re-run: Parts 0–1 use only their saved probabilities.

**Baselines,** all on the same calibrated probabilities and the same outer test days:
- I alone, F1 alone, and A alone (for context);
- **EQUAL:** 1/3 weight each;
- **TUNED:** one fixed weight triple, fit on ROUTER_HALF by NLL over a 0.05-step simplex grid;
- **ORACLE:** correct if any expert is correct.

In [31]:
# ==============================================================================
# 5. ROUTER
# ==============================================================================
import torch
import torch.nn as nn

torch.use_deterministic_algorithms(True)
PAIRS = [("A", "I"), ("A", "F1"), ("I", "F1")]


def features(qs):
    """qs: dict expert -> (n, 5) calibrated probabilities. Returns (n, 27)."""
    f = [qs[m] for m in EXPERTS]
    f += [(-(qs[m] * np.log(np.clip(qs[m], 1e-12, None))).sum(1, keepdims=True)) / math.log(N_CLASSES) for m in EXPERTS]
    f += [qs[m].max(1, keepdims=True) for m in EXPERTS]
    for a, b in PAIRS:
        f.append(np.abs(qs[a] - qs[b]).sum(1, keepdims=True))
        f.append((qs[a].argmax(1) == qs[b].argmax(1)).astype(np.float64)[:, None])
    return np.concatenate(f, axis=1)


N_FEAT = features({m: np.full((1, N_CLASSES), 0.2) for m in EXPERTS}).shape[1]
print(f"Router input dimension: {N_FEAT} = 15 (3 x 5 probabilities) + 3 (entropies) + 3 (max probs) + 6 (3 pairs x [L1, agree])")
assert N_FEAT == 27


def make_router():
    return nn.Sequential(nn.Linear(N_FEAT, ROUTER_HIDDEN), nn.ReLU(), nn.Linear(ROUTER_HIDDEN, len(EXPERTS)))


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


ROUTER_PARAMS = sum(p.numel() for p in make_router().parameters())
print(f"Router parameters: {ROUTER_PARAMS:,} (Linear {N_FEAT}->{ROUTER_HIDDEN}, ReLU, Linear {ROUTER_HIDDEN}->3)")
_pc = pd.concat([pd.read_csv(V29_RUNS_CSV).groupby("model").n_params.first(),
                 pd.read_csv(V30_RUNS_CSV).groupby("model").n_params.first() if V30_RUNS_CSV.exists() else pd.Series(dtype=float)])
print("Parameter counts (experts from the saved CSVs; they are NOT instantiated here):")
for name, n in list(_pc.items()) + [("ROUTER", ROUTER_PARAMS)]:
    print(f"    {name:<7} {int(n):>11,}")


def train_router(X_tr, qs_tr, y_tr, seed):
    seed_all(seed)
    net = make_router()
    opt = torch.optim.Adam(net.parameters(), lr=ROUTER_LR, weight_decay=ROUTER_WD)
    X = torch.tensor(X_tr, dtype=torch.float32)
    E = torch.tensor(np.stack([qs_tr[m] for m in EXPERTS], axis=1), dtype=torch.float32)     # (n, 3, 5)
    yt = torch.tensor(y_tr, dtype=torch.long)
    losses = []
    for ep in range(ROUTER_EPOCHS):
        opt.zero_grad()
        w = torch.softmax(net(X), dim=-1)                                                   # (n, 3)
        p = (w.unsqueeze(-1) * E).sum(1)                                                    # (n, 5)
        loss = -torch.log(p[torch.arange(len(yt)), yt].clamp_min(1e-12)).mean()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return net, losses


def router_predict(net, X, qs):
    with torch.no_grad():
        w = torch.softmax(net(torch.tensor(X, dtype=torch.float32)), dim=-1).numpy().astype(np.float64)
    p = sum(w[:, i:i + 1] * qs[m] for i, m in enumerate(EXPERTS))
    return p, w


SIMPLEX = [np.array(w) for w in itertools.product(np.round(np.arange(0, 1 + 1e-9, TUNED_GRID_STEP), 4), repeat=3)
           if abs(sum(w) - 1) < 1e-9]
METHODS = ["A (cal)", "I (cal)", "F1 (cal)", "EQUAL", "TUNED", "ROUTER", "ORACLE"]
OUT = {m: {s: np.zeros((len(META), N_CLASSES)) for s in SEEDS} for m in METHODS}   # test probabilities per method/seed
WEIGHTS = {s: np.zeros((len(META), 3)) for s in SEEDS}
TUNED_W, ROUTER_LOSS = {}, {}
_t = time.time()
for k in range(len(DAYS)):
    sp = SPLITS[k]
    for s in SEEDS:
        qs = {m: Q[(m, k, s)] for m in EXPERTS}
        X_all = features(qs)
        r_idx, t_idx = sp["router"], sp["test"]
        mu, sd = X_all[r_idx].mean(0), X_all[r_idx].std(0) + 1e-6
        Xz = (X_all - mu) / sd
        hook("features", np.isfinite(Xz[r_idx]).all() and np.isfinite(Xz[t_idx]).all() and Xz.shape[1] == N_FEAT,
             f"fold {k} seed {s}: router input {Xz.shape[1]} dims, no NaN/Inf (ROUTER_HALF {len(r_idx)}, test {len(t_idx)})")
        # single experts
        for m in EXPERTS:
            OUT[f"{m} (cal)"][s][t_idx] = qs[m][t_idx]
        # EQUAL
        OUT["EQUAL"][s][t_idx] = np.mean([qs[m][t_idx] for m in EXPERTS], axis=0)
        # TUNED: one weight triple by NLL on ROUTER_HALF
        best = min(SIMPLEX, key=lambda w: nll(sum(w[i] * qs[m][r_idx] for i, m in enumerate(EXPERTS)), Y[r_idx]))
        TUNED_W[(k, s)] = best
        OUT["TUNED"][s][t_idx] = sum(best[i] * qs[m][t_idx] for i, m in enumerate(EXPERTS))
        # ROUTER
        net, losses = train_router(Xz[r_idx], {m: qs[m][r_idx] for m in EXPERTS}, Y[r_idx], s)
        ROUTER_LOSS[(k, s)] = (losses[0], losses[-1])
        p_t, w_t = router_predict(net, Xz[t_idx], {m: qs[m][t_idx] for m in EXPERTS})
        OUT["ROUTER"][s][t_idx] = p_t
        WEIGHTS[s][t_idx] = w_t
        # ORACLE (per window): true class if any expert's argmax is right, else the EQUAL argmax (one-hot)
        preds = np.stack([qs[m][t_idx].argmax(1) for m in EXPERTS])
        any_ok = (preds == Y[t_idx]).any(0)
        op = np.where(any_ok, Y[t_idx], OUT["EQUAL"][s][t_idx].argmax(1))
        OUT["ORACLE"][s][t_idx] = np.eye(N_CLASSES)[op]
        print(f"fold {k} seed {s}: TUNED weights (A, I, F1) = {tuple(np.round(best, 2))} | router CE {losses[0]:.4f} -> "
              f"{losses[-1]:.4f} on ROUTER_HALF | T (A, I, F1) = "
              f"{tuple(round(CAL_T[(m, k, s)], 3) for m in EXPERTS)}")
print(f"All routers and baselines: {time.time() - _t:.1f} s")

Router input dimension: 27 = 15 (3 x 5 probabilities) + 3 (entropies) + 3 (max probs) + 6 (3 pairs x [L1, agree])
Router parameters: 995 (Linear 27->32, ReLU, Linear 32->3)
Parameter counts (experts from the saved CSVs; they are NOT instantiated here):
    A         1,012,358
    D         1,692,884
    F0        1,067,554
    F1        1,100,516
    F2        1,133,348
    I           977,350
    CA        1,693,222
    CAG       1,726,184
    ROUTER          995
[HOOK features] PASS — fold 0 seed 42: router input 27 dims, no NaN/Inf (ROUTER_HALF 2391, test 602)
fold 0 seed 42: TUNED weights (A, I, F1) = (np.float64(0.0), np.float64(0.6), np.float64(0.4)) | router CE 0.4182 -> 0.1794 on ROUTER_HALF | T (A, I, F1) = (3.475, 2.297, 3.314)
[HOOK features] PASS — fold 0 seed 123: router input 27 dims, no NaN/Inf (ROUTER_HALF 2391, test 602)
fold 0 seed 123: TUNED weights (A, I, F1) = (np.float64(0.0), np.float64(0.85), np.float64(0.15)) | router CE 0.4054 -> 0.1838 on ROUTER_HALF | T (A, 

## 6. Results — router and baselines, pooled out-of-fold (window and mission level)
Each method's test probabilities are averaged over seeds 42 and 123 per window. Every window is predicted by the
model for its own unseen day. The metrics use v29's own functions: accuracy, macro-F1 with a 95 % mission-level
bootstrap CI (2000 resamples within class), balanced accuracy, macro-AUROC, per-class recall, and the safety
breakdown.

**ORACLE** is correct whenever any expert is correct. At mission level it uses each expert's mission-level prediction.
It is an unreachable ceiling, and its AUROC is not meaningful (reported as NaN).

**Success criterion (stated in advance):** the realistic bar is **I alone at window level**, pooled window macro-F1
≈ 0.893. Mission-level numbers are context only. **Beating I's mission macro-F1 (≈ 0.968, with a wide CI) is NOT the
success criterion.**

> ⚠️ Every Part 1 number below carries the header's caveats. **CAL_HALF / ROUTER_HALF come from the other four days'
> saved test predictions** (v29 saved no inner-validation probabilities). The router is **not** a clean nested
> cross-validation.

In [32]:
# ==============================================================================
# 6. RESULTS TABLE
# ==============================================================================
CAVEAT = ("CAVEAT: calibration/router data = " + SPLIT_SOURCE + " (Section 0, like v29, saves test-fold probabilities only); "
          "experts are the Section 0 checkpoints (v29's code), fixed after training; the router is not a clean nested CV "
          "(no nested expert retraining). Day k itself "
          "is never used for calibration or router training.")
POOLED = {m: np.mean([OUT[m][s] for s in SEEDS], axis=0) for m in METHODS if m != "ORACLE"}
# pooled oracle from the seed-averaged experts (averaging two one-hot oracle vectors would create ties)
_pe = np.stack([POOLED[f"{e} (cal)"].argmax(1) for e in EXPERTS])
POOLED["ORACLE"] = np.eye(N_CLASSES)[np.where((_pe == Y).any(0), Y, POOLED["EQUAL"].argmax(1))]


def oracle_mission(prob_by_expert):
    """Mission-level oracle from each expert's mission-level prediction (argmax of its mean window probability)."""
    preds, ym, names = {}, None, None
    for m, p in prob_by_expert.items():
        ym, pm, names = mission_level(META.mission.values, Y, p)
        preds[m] = pm.argmax(1)
    any_ok = np.any([preds[m] == ym for m in preds], axis=0)
    eq = mission_level(META.mission.values, Y, np.mean(list(prob_by_expert.values()), axis=0))[1].argmax(1)
    return ym, np.eye(N_CLASSES)[np.where(any_ok, ym, eq)], names


RESULTS = []
for m in METHODS:
    for level in ("window", "mission"):
        if level == "window":
            b = level_block(META.mission.values, Y, POOLED[m])
        elif m == "ORACLE":
            ym, pm, names = oracle_mission({e: POOLED[f"{e} (cal)"] for e in EXPERTS})
            b = level_block(names, ym, pm)
        else:
            ym, pm, names = mission_level(META.mission.values, Y, POOLED[m])
            b = level_block(names, ym, pm)
        if m == "ORACLE":
            b["macro_auroc"] = float("nan")
        RESULTS.append({"method": m, "level": level, **{k_: v for k_, v in b.items() if k_ != "cm"}})
RESULTS = pd.DataFrame(RESULTS)
RESULTS.to_csv(WORK / "v32_results_table.csv", index=False)
pd.set_option("display.width", 250)
for level in ("window", "mission"):
    t = RESULTS[RESULTS.level == level].set_index("method")
    t["macro-F1 [95% CI]"] = [f"{a:.4f} [{b:.4f}, {c:.4f}]" for a, b, c in zip(t.macro_f1, t.f1_ci_low, t.f1_ci_high)]
    print(f"\n=== POOLED DAY-OUT — {level.upper()} LEVEL (seed-averaged; 5 folds x seeds {SEEDS}) ===")
    print(t[["accuracy", "macro-F1 [95% CI]", "balanced_accuracy", "macro_auroc"] + [f"recall_c{c}" for c in range(N_CLASSES)]
            + ["faulty_pred_normal", "normal_pred_faulty", "adjacent_error_share"]].round(4).to_string())
print("\nSuccess criterion: ROUTER vs I (cal) at WINDOW level (I ≈ 0.893). Mission level is context only.")
print(CAVEAT)
print("Note: calibration preserves each window's argmax, so 'I (cal)' / 'F1 (cal)' window-level accuracy and macro-F1 equal "
      "v29's I / F1 values; only mixtures and mission-level averages change.")


=== POOLED DAY-OUT — WINDOW LEVEL (seed-averaged; 5 folds x seeds [42, 123]) ===
          accuracy        macro-F1 [95% CI]  balanced_accuracy  macro_auroc  recall_c0  recall_c1  recall_c2  recall_c3  recall_c4  faulty_pred_normal  normal_pred_faulty  adjacent_error_share
method                                                                                                                                                                                          
A (cal)     0.7967  0.7995 [0.7436, 0.8525]             0.7987       0.9381      1.000     0.8492     0.6458     0.7102     0.7883              0.0007               0.000                0.9934
I (cal)     0.8961  0.8950 [0.8665, 0.9199]             0.8963       0.9871      0.999     0.8976     0.7813     0.8542     0.9496              0.0267               0.001                0.8456
F1 (cal)    0.8675  0.8678 [0.8195, 0.9123]             0.8679       0.9626      1.000     0.8757     0.7290     0.8175     0.9175              0.

## 7. Paired comparisons across the 5 folds — both methods (as in v30)
- **Pooled-probability method:** fold macro-F1 of the seed-averaged probabilities.
- **Seed-averaged-F1 method:** the mean over seeds of each seed's own fold macro-F1.

Both are available for every method here, because all probabilities are in memory. The table gives mean Δ, SD, wins
out of 5 and a paired t-test (df 4), at window and mission level. Any disagreement between the methods in win count or
sign is flagged.

The verdict rule is v29's, fixed in advance: a method counts as "BEATS" only with mean Δ > 0.016 **and** at least
4 of 5 wins.

In [33]:
# ==============================================================================
# 7. PAIRED COMPARISONS
# ==============================================================================
def fold_f1(probs, level, method):
    out = []
    for k, day in enumerate(DAYS):
        sel = META.day.values == day
        if level == "window":
            out.append(f1_score(Y[sel], probs[sel].argmax(1), average="macro", labels=LABELS, zero_division=0))
        elif method == "ORACLE":
            ym, pm, _ = oracle_mission_fold(sel, probs)
            out.append(f1_score(ym, pm.argmax(1), average="macro", labels=LABELS, zero_division=0))
        else:
            ym, pm, _ = mission_level(META.mission.values[sel], Y[sel], probs[sel])
            out.append(f1_score(ym, pm.argmax(1), average="macro", labels=LABELS, zero_division=0))
    return np.array(out)


def oracle_mission_fold(sel, experts_probs):
    preds, ym, names = {}, None, None
    for e, p in experts_probs.items():
        ym, pm, names = mission_level(META.mission.values[sel], Y[sel], p[sel])
        preds[e] = pm.argmax(1)
    any_ok = np.any([preds[e] == ym for e in preds], axis=0)
    eq = mission_level(META.mission.values[sel], Y[sel], np.mean([p[sel] for p in experts_probs.values()], axis=0))[1].argmax(1)
    return ym, np.eye(N_CLASSES)[np.where(any_ok, ym, eq)], names


def scores(method, level):
    """(pooled-probability fold scores, seed-averaged fold scores)."""
    if method == "ORACLE" and level == "mission":
        pooled = fold_f1({e: POOLED[f"{e} (cal)"] for e in EXPERTS}, level, method)
        per_seed = [fold_f1({e: OUT[f"{e} (cal)"][s] for e in EXPERTS}, level, method) for s in SEEDS]
    else:
        pooled = fold_f1(POOLED[method], level, method)
        per_seed = [fold_f1(OUT[method][s], level, method) for s in SEEDS]
    return pooled, np.mean(per_seed, axis=0)


def verdict(st):
    return "BEATS" if st["mean"] > NOISE_FLOOR_F1 and st["wins"] >= 4 else (
        "WORSE" if st["mean"] < -NOISE_FLOOR_F1 and st["wins"] <= 1 else "no reliable difference")


PAIRED = []
for other in ["I (cal)", "F1 (cal)", "EQUAL", "TUNED", "ORACLE"]:
    for level in ("window", "mission"):
        (pa, sa), (pb, sb) = scores("ROUTER", level), scores(other, level)
        row = {"comparison": f"ROUTER − {other}", "level": level}
        for meth, (a, b) in (("pooled-probability", (pa, pb)), ("seed-averaged-F1", (sa, sb))):
            d = a - b
            t_, p_ = stats.ttest_rel(a, b)
            row[meth] = {"mean": d.mean(), "sd": d.std(ddof=1), "wins": int((d > 0).sum()), "p": p_, "diffs": np.round(d, 4).tolist()}
        pp, sf = row["pooled-probability"], row["seed-averaged-F1"]
        row["flag"] = ("⚠️ METHOD DISAGREEMENT (wins or sign differ) — both shown" if pp["wins"] != sf["wins"] or
                       np.sign(pp["mean"]) != np.sign(sf["mean"]) else "methods agree")
        PAIRED.append(row)
        print(f"\n{row['comparison']} [{level}]")
        for meth in ("pooled-probability", "seed-averaged-F1"):
            st = row[meth]
            print(f"   {meth:<20} mean Δ {st['mean']:+.4f}  SD {st['sd']:.4f}  wins {st['wins']}/5  p {st['p']:.3f}  "
                  f"per fold {st['diffs']}  -> {verdict(st)}")
        print(f"   {row['flag']}")
print("\n" + CAVEAT)


ROUTER − I (cal) [window]
   pooled-probability   mean Δ -0.0018  SD 0.0467  wins 3/5  p 0.934  per fold [0.0185, -0.0075, 0.0094, 0.0478, -0.0773]  -> no reliable difference
   seed-averaged-F1     mean Δ -0.0016  SD 0.0507  wins 3/5  p 0.949  per fold [0.0162, -0.0124, 0.0135, 0.056, -0.0811]  -> no reliable difference
   methods agree

ROUTER − I (cal) [mission]
   pooled-probability   mean Δ -0.0657  SD 0.0738  wins 0/5  p 0.118  per fold [0.0, -0.0537, -0.1, 0.0, -0.1745]  -> WORSE
   seed-averaged-F1     mean Δ -0.0462  SD 0.0575  wins 0/5  p 0.147  per fold [0.0, -0.0404, -0.05, 0.0, -0.1406]  -> WORSE
   methods agree

ROUTER − F1 (cal) [window]
   pooled-probability   mean Δ +0.0068  SD 0.0283  wins 3/5  p 0.622  per fold [-0.034, 0.0422, 0.0172, -0.0049, 0.0133]  -> no reliable difference
   seed-averaged-F1     mean Δ +0.0000  SD 0.0234  wins 2/5  p 0.999  per fold [-0.0279, 0.023, 0.0235, -0.0183, -0.0002]  -> no reliable difference
   ⚠️ METHOD DISAGREEMENT (wins or sign 

## 8. Router behaviour diagnostics
- **Weight distribution:** mean, SD, min and max of (w_A, w_I, w_F) over the test windows, per seed and per fold.
- **Collapse check:** a weight counts as collapsed to a near-constant if its across-window SD is below 0.05.
- **Mechanism test:** the router is supposed to trust an expert more when that expert is confident, and when it is
  right. The test is the Spearman correlation between each expert's weight and that expert's own calibrated max
  probability, and between the weight and whether that expert is correct.

In [34]:
# ==============================================================================
# 8. ROUTER DIAGNOSTICS
# ==============================================================================
DIAG = []
for s in SEEDS:
    W = WEIGHTS[s]
    for i, e in enumerate(EXPERTS):
        conf = np.zeros(len(META))
        corr_ok = np.zeros(len(META))
        for k in range(len(DAYS)):
            t_idx = SPLITS[k]["test"]
            conf[t_idx] = Q[(e, k, s)][t_idx].max(1)
            corr_ok[t_idx] = (Q[(e, k, s)][t_idx].argmax(1) == Y[t_idx])
        rho_c, p_c = stats.spearmanr(W[:, i], conf)
        rho_k, p_k = stats.spearmanr(W[:, i], corr_ok)
        DIAG.append({"seed": s, "expert": e, "w mean": W[:, i].mean(), "w sd": W[:, i].std(), "w min": W[:, i].min(),
                     "w max": W[:, i].max(), "collapsed (sd<0.05)": W[:, i].std() < COLLAPSE_SD,
                     "rho(w, own max prob)": rho_c, "p conf": p_c, "rho(w, own correct)": rho_k, "p correct": p_k})
DIAG = pd.DataFrame(DIAG)
print(DIAG.round(4).to_string(index=False))
print("\nPer fold mean weights (seed-averaged):")
_Wm = np.mean([WEIGHTS[s] for s in SEEDS], axis=0)
print(pd.DataFrame([{"fold": k, "day": d, **{f"w_{e}": _Wm[META.day.values == d, i].mean() for i, e in enumerate(EXPERTS)}}
                    for k, d in enumerate(DAYS)]).set_index("fold").round(3).to_string())
_supp = DIAG[(DIAG["rho(w, own max prob)"] > 0) & (DIAG["p conf"] < 0.05)]
_collapsed = bool(DIAG.groupby("seed")["collapsed (sd<0.05)"].all().any())
MECH_VERDICT = (
    ("Weights COLLAPSE to a near-constant mix in at least one seed (every weight's sd < 0.05): the router then behaves like a "
     "fixed weighting, not per-window routing. " if _collapsed else "Weights vary across windows (not collapsed). ")
    + (f"The weight given to an expert rises with that expert's own confidence in {len(_supp)}/{len(DIAG)} (expert, seed) cases "
       f"(Spearman > 0, p < 0.05)" + (" — consistent with the intended mechanism." if len(_supp) == len(DIAG) else
                                      " — only PARTIAL support for the intended mechanism; see the table for which experts/seeds.")))
print("\nMECHANISM: " + MECH_VERDICT)

 seed expert  w mean   w sd  w min  w max  collapsed (sd<0.05)  rho(w, own max prob)  p conf  rho(w, own correct)  p correct
   42      A  0.0224 0.1033    0.0 0.9965                False                0.3249     0.0               0.2386     0.0000
   42      I  0.4449 0.4582    0.0 1.0000                False                0.4363     0.0               0.0889     0.0000
   42     F1  0.5328 0.4553    0.0 1.0000                False                0.3177     0.0              -0.1337     0.0000
  123      A  0.1760 0.3404    0.0 1.0000                False                0.5641     0.0              -0.0137     0.3226
  123      I  0.6206 0.4379    0.0 1.0000                False                0.4997     0.0               0.1670     0.0000
  123     F1  0.2033 0.3517    0.0 1.0000                False                0.2077     0.0              -0.1561     0.0000

Per fold mean weights (seed-averaged):
             day    w_A    w_I   w_F1
fold                                 
0     202

## 9. Nested cross-validation — future work, not an option in this notebook
A clean version of Part 1 would need **expert** predictions on inner-validation windows produced by checkpoints that
were never selected on them. That means one of two things:
- nested cross-validation, where each outer fold's experts are retrained with a held-out router split;
- or at least re-saving v29's inner-validation probabilities.

Both require an **additional, nested level of retraining A, I and F1**. Section 0 contains the training code, but
nested retraining is not implemented, so there is deliberately no `RUN_NESTED_CV` switch in this notebook.

Occlusion and other attribution or explainability analyses are also deferred to future work.

In [35]:
# ==============================================================================
# 10. SUMMARY + HOOKS
# ==============================================================================
def row(method, level):
    return RESULTS[(RESULTS.method == method) & (RESULTS.level == level)].iloc[0]


def comp(name, level="window"):
    r = next(x for x in PAIRED if x["comparison"] == name and x["level"] == level)
    return (f"{name} [{level}]: pooled-probability Δ {r['pooled-probability']['mean']:+.4f} (wins {r['pooled-probability']['wins']}/5, "
            f"p {r['pooled-probability']['p']:.3f}) -> {verdict(r['pooled-probability'])} | seed-averaged-F1 Δ "
            f"{r['seed-averaged-F1']['mean']:+.4f} (wins {r['seed-averaged-F1']['wins']}/5, p {r['seed-averaged-F1']['p']:.3f}) -> "
            f"{verdict(r['seed-averaged-F1'])}  [{r['flag']}]")


_rw, _iw, _tw = row("ROUTER", "window"), row("I (cal)", "window"), row("TUNED", "window")
_ri = next(x for x in PAIRED if x["comparison"] == "ROUTER − I (cal)" and x["level"] == "window")
_rt = next(x for x in PAIRED if x["comparison"] == "ROUTER − TUNED" and x["level"] == "window")
_beats_i = all(verdict(_ri[m]) == "BEATS" for m in ("pooled-probability", "seed-averaged-F1"))
_beats_t = all(verdict(_rt[m]) == "BEATS" for m in ("pooled-probability", "seed-averaged-F1"))
L = ["WHAT v32 TELLS US", "",
     f"Data: experts A, I, F1 TRAINED IN SECTION 0 of this notebook (v29's code, 5 folds x seeds {SEEDS}); Parts 0-1 read "
     f"their per-window test probabilities from {NPZ_DIR} (no logits, no inner-validation windows) plus the v29/v30 CSVs.",
     f"CAL_HALF / ROUTER_HALF source: {SPLIT_SOURCE}.", CAVEAT, "",
     "PART 0: " + PART0_STATEMENT, "",
     "Calibration: " + f"{int(CAL_TAB.reduced.sum())}/{len(CAL_TAB)} expert/fold/seed fits did not increase CAL_HALF NLL; mean "
     f"NLL {CAL_TAB['NLL before'].mean():.4f} -> {CAL_TAB['NLL after'].mean():.4f}; mean T " + ", ".join(
         f"{e} {CAL_TAB[CAL_TAB.expert == e]['T'].mean():.2f}" for e in EXPERTS) + ".", "",
     "Pooled window macro-F1 [95% CI]: " + "; ".join(
         f"{m} {row(m, 'window').macro_f1:.3f} [{row(m, 'window').f1_ci_low:.3f}, {row(m, 'window').f1_ci_high:.3f}]" for m in METHODS),
     "Pooled mission macro-F1 (context): " + "; ".join(f"{m} {row(m, 'mission').macro_f1:.3f}" for m in METHODS), "",
     "Paired (5 folds, both methods):"] + ["  " + comp(f"ROUTER − {o}") for o in ["I (cal)", "F1 (cal)", "EQUAL", "TUNED", "ORACLE"]] + [
     "", "Router behaviour: " + MECH_VERDICT, "",
     "VERDICT: " + (
         f"the router {'BEATS' if _beats_i else 'does NOT reliably beat'} I alone at window level (router {_rw.macro_f1:.3f} vs I "
         f"{_iw.macro_f1:.3f}) and {'BEATS' if _beats_t else 'does NOT reliably beat'} TUNED fixed-weight averaging "
         f"({_tw.macro_f1:.3f}) under the pre-set rule (mean Δ > {NOISE_FLOOR_F1} and >= 4/5 wins, required under BOTH methods). "
         + ("Per-instance routing is therefore credited with a gain beyond fixed weighting." if _beats_t else
            "Any gain over I is not attributable to per-instance routing specifically unless it also beats TUNED.")),
     "", "Deferred: nested CV with nested expert retraining, occlusion / attribution analyses.",
     "Context — " + ("D/F0/F2 trained in Section 0 (window macro-F1): " + "; ".join(
         f"{m} {SECTION0_TABLE.loc[m, 'window macro-F1 [95% CI]']}" for m in ("D", "F0", "F2")) if TRAIN_EXTRA_MODELS else
         "HARDCODED REFERENCE VALUES, NOT REPRODUCED IN THIS SESSION, from a prior run (window macro-F1; not used in any "
         "computation): D 0.871 | F0 0.838 | F2 0.853"),
     f"Total notebook time: {(time.time() - T0_NOTEBOOK) / 60:.1f} min."]
SUMMARY = "\n".join(L)
print(SUMMARY)
(WORK / "v32_summary.txt").write_text(SUMMARY, encoding="utf-8")

print("\n" + "=" * 90 + "\nVERIFICATION HOOKS\n" + "=" * 90)
for name, res in list(SECTION0_HOOKS.items()) + list(HOOKS.items()):
    print(f"  {name:<12} {'PASS' if all(ok for ok, _ in res) else 'FAIL'}  ({len(res)} checks)")
    for ok, msg in res:
        if not ok:
            print(f"      FAIL: {msg}")

WHAT v32 TELLS US

Data: experts A, I, F1 TRAINED IN SECTION 0 of this notebook (v29's code, 5 folds x seeds [42, 123]); Parts 0-1 read their per-window test probabilities from /kaggle/working/v32_npz (no logits, no inner-validation windows) plus the v29/v30 CSVs.
CAL_HALF / ROUTER_HALF source: FALLBACK: other four days' saved test windows.
CAVEAT: calibration/router data = FALLBACK: other four days' saved test windows (Section 0, like v29, saves test-fold probabilities only); experts are the Section 0 checkpoints (v29's code), fixed after training; the router is not a clean nested CV (no nested expert retraining). Day k itself is never used for calibration or router training.

PART 0: Window level: the A+I oracle already reaches 0.954 accuracy; adding F1 raises the ceiling by +0.007 (to 0.961). F1's errors are largely REDUNDANT with I's (P(F1 wrong | I wrong) = 0.42 vs P(F1 wrong | I right) = 0.10). F1 adds little headroom beyond A+I as a third expert (< 0.01), so any router gain must